In [ ]:
!pip -q install imbalanced-learn

In [ ]:
import warnings, glob
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler

In [ ]:
# =========================
# LOAD + CLEAN + BUILD PANEL
# =========================

candidate_files = sorted(glob.glob("ShootingVictim 2022-2024*.csv"))
if not candidate_files:
    raise FileNotFoundError("Upload 'ShootingVictim 2022-2024.csv' first.")

FILE_NAME = candidate_files[-1]
print("Using file:", FILE_NAME)

df = pd.read_csv(FILE_NAME)
print("Raw shape:", df.shape)

df["OCCUR_DATE"] = pd.to_datetime(df["OCCUR_DATE"], errors="coerce")
df["PRECINCT"] = pd.to_numeric(df["PRECINCT"], errors="coerce")

df = df.dropna(subset=["OCCUR_DATE", "PRECINCT", "BORO"]).copy()
df = df[df["PRECINCT"] > 0].copy()

for c in ["victims_in_incident", "multi_victim_flag", "mass_casualty_flag", "incident_murder_flag"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    else:
        df[c] = 0

df["week_start"] = df["OCCUR_DATE"] - pd.to_timedelta(df["OCCUR_DATE"].dt.weekday, unit="D")

weekly = (
    df.groupby(["PRECINCT", "BORO", "week_start"], as_index=False)
      .agg(
          shootings_count=("INCIDENT_KEY", "count"),
          total_victims_in_week=("victims_in_incident", "sum"),
          max_victims_in_incident=("victims_in_incident", "max"),
          num_multi_victim_incidents=("multi_victim_flag", "sum"),
          any_multi_victim_week=("multi_victim_flag", "max"),
          num_fatal_incidents=("incident_murder_flag", "sum"),
          any_fatal_week=("incident_murder_flag", "max")
      )
)

all_weeks = pd.date_range(df["week_start"].min(), df["week_start"].max(), freq="W-MON")
precinct_boro = df[["PRECINCT", "BORO"]].drop_duplicates()

panel = (
    precinct_boro.assign(key=1)
    .merge(pd.DataFrame({"week_start": all_weeks, "key": 1}), on="key")
    .drop(columns="key")
    .merge(weekly, on=["PRECINCT", "BORO", "week_start"], how="left")
)

fill_cols = [c for c in panel.columns if c not in ["PRECINCT", "BORO", "week_start"]]
panel[fill_cols] = panel[fill_cols].fillna(0)

panel["is_hotspot"] = (
    (panel["shootings_count"] >= 2) |
    (panel["any_fatal_week"] == 1) |
    (panel["max_victims_in_incident"] >= 2)
).astype(int)

print("Panel shape:", panel.shape)
print("Hotspot rate:", round(panel["is_hotspot"].mean(), 4))
print(panel["is_hotspot"].value_counts())

Using file: ShootingVictim 2022-2024.csv
Raw shape: (3167, 27)
Panel shape: (12008, 11)
Hotspot rate: 0.0925
is_hotspot
0    10897
1     1111
Name: count, dtype: int64


In [ ]:
# =========================
# FEATURE ENGINEERING
# =========================

panel = panel.sort_values(["PRECINCT", "week_start"]).reset_index(drop=True)

panel["month"] = panel["week_start"].dt.month
panel["week_of_year"] = panel["week_start"].dt.isocalendar().week.astype(int)

g = panel.groupby("PRECINCT")

panel["lag_1wk"] = g["shootings_count"].shift(1)
panel["lag_2wk"] = g["shootings_count"].shift(2)
panel["prev_4wk_sum"] = g["shootings_count"].shift(1).rolling(4).sum().reset_index(level=0, drop=True)
panel["prev_4wk_avg"] = g["shootings_count"].shift(1).rolling(4).mean().reset_index(level=0, drop=True)

panel["prev_spike_flag"] = (panel["lag_1wk"] > 1.5 * panel["prev_4wk_avg"]).astype(int)

panel["lag_1wk_hot"] = g["is_hotspot"].shift(1)
panel["prev_4wk_hot_count"] = g["is_hotspot"].shift(1).rolling(4).sum().reset_index(level=0, drop=True)

panel["lag_1wk_total_victims_in_week"] = g["total_victims_in_week"].shift(1)
panel["prev_4wk_num_fatal_incidents"] = g["num_fatal_incidents"].shift(1).rolling(4).sum().reset_index(level=0, drop=True)
panel["prev_4wk_num_multi_victim_incidents"] = g["num_multi_victim_incidents"].shift(1).rolling(4).sum().reset_index(level=0, drop=True)

panel["fatal_share_prev_4wk"] = (
    panel["prev_4wk_num_fatal_incidents"] / panel["prev_4wk_sum"].replace(0, np.nan)
)

panel = panel.replace([np.inf, -np.inf], np.nan)

selected_features = [
    "PRECINCT",
    "BORO",
    "month",
    "week_of_year",
    "lag_1wk",
    "lag_2wk",
    "prev_4wk_sum",
    "prev_4wk_avg",
    "prev_spike_flag",
    "lag_1wk_hot",
    "prev_4wk_hot_count",
    "lag_1wk_total_victims_in_week",
    "prev_4wk_num_fatal_incidents",
    "prev_4wk_num_multi_victim_incidents",
    "fatal_share_prev_4wk"
]

model_df = panel.copy()
model_df = model_df[
    model_df["week_start"] >= model_df["week_start"].min() + pd.Timedelta(weeks=4)
].copy()

X = model_df[selected_features].copy()
y = model_df["is_hotspot"].astype(int).copy()

unique_weeks = np.array(sorted(model_df["week_start"].unique()))
n_weeks = len(unique_weeks)

train_end = int(n_weeks * 0.60)
val_end = int(n_weeks * 0.80)

train_weeks = unique_weeks[:train_end]
val_weeks = unique_weeks[train_end:val_end]
test_weeks = unique_weeks[val_end:]

train_idx = model_df["week_start"].isin(train_weeks)
val_idx = model_df["week_start"].isin(val_weeks)
test_idx = model_df["week_start"].isin(test_weeks)

X_train_df, y_train = X.loc[train_idx].copy(), y.loc[train_idx].copy()
X_val_df, y_val = X.loc[val_idx].copy(), y.loc[val_idx].copy()
X_test_df, y_test = X.loc[test_idx].copy(), y.loc[test_idx].copy()

print("Train shape:", X_train_df.shape, "| hotspot rate:", round(y_train.mean(), 4))
print("Val shape:  ", X_val_df.shape, "| hotspot rate:", round(y_val.mean(), 4))
print("Test shape: ", X_test_df.shape, "| hotspot rate:", round(y_test.mean(), 4))

Train shape: (6992, 15) | hotspot rate: 0.1028
Val shape:   (2356, 15) | hotspot rate: 0.0709
Test shape:  (2356, 15) | hotspot rate: 0.0862


In [ ]:
# =========================
# HELPERS
# =========================

numeric_features = [c for c in selected_features if c not in ["PRECINCT", "BORO"]]

def fill_numeric(train_df, other_df, cols):
    imputer = SimpleImputer(strategy="median")
    train_num = pd.DataFrame(imputer.fit_transform(train_df[cols]), columns=cols, index=train_df.index)
    other_num = pd.DataFrame(imputer.transform(other_df[cols]), columns=cols, index=other_df.index)
    return train_num, other_num

def build_threshold_guess_features(train_df, other_df, numeric_cols, cat_cols, quantiles=(0.25, 0.5, 0.75)):
    train_num, other_num = fill_numeric(train_df, other_df, numeric_cols)

    train_parts = []
    other_parts = []

    for col in numeric_cols:
        cutoffs = np.unique(np.quantile(train_num[col], quantiles))
        for cutoff in cutoffs:
            cname = f"{col}>={round(float(cutoff), 4)}"
            train_parts.append((train_num[col] >= cutoff).astype(int).rename(cname))
            other_parts.append((other_num[col] >= cutoff).astype(int).rename(cname))

    train_bin = pd.concat(train_parts, axis=1)
    other_bin = pd.concat(other_parts, axis=1)

    train_cat = pd.get_dummies(train_df[cat_cols].astype(str), prefix=cat_cols)
    other_cat = pd.get_dummies(other_df[cat_cols].astype(str), prefix=cat_cols)

    other_cat = other_cat.reindex(columns=train_cat.columns, fill_value=0)

    train_out = pd.concat([train_bin, train_cat], axis=1)
    other_out = pd.concat([other_bin, other_cat], axis=1)

    other_out = other_out.reindex(columns=train_out.columns, fill_value=0)
    return train_out, other_out

def build_scaled_features(train_df, other_df, numeric_cols, cat_cols):
    train_num, other_num = fill_numeric(train_df, other_df, numeric_cols)

    scaler = StandardScaler()
    train_num_scaled = pd.DataFrame(
        scaler.fit_transform(train_num),
        columns=numeric_cols,
        index=train_df.index
    )
    other_num_scaled = pd.DataFrame(
        scaler.transform(other_num),
        columns=numeric_cols,
        index=other_df.index
    )

    train_cat = pd.get_dummies(train_df[cat_cols].astype(str), prefix=cat_cols)
    other_cat = pd.get_dummies(other_df[cat_cols].astype(str), prefix=cat_cols)
    other_cat = other_cat.reindex(columns=train_cat.columns, fill_value=0)

    train_out = pd.concat([train_num_scaled, train_cat], axis=1)
    other_out = pd.concat([other_num_scaled, other_cat], axis=1)
    other_out = other_out.reindex(columns=train_out.columns, fill_value=0)

    return train_out, other_out

def choose_threshold(y_true, pred_proba):
    thresholds = np.arange(0.10, 0.91, 0.05)
    rows = []

    for t in thresholds:
        pred = (pred_proba >= t).astype(int)
        rows.append({
            "threshold": t,
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
            "f1": f1_score(y_true, pred, zero_division=0)
        })

    thr_df = pd.DataFrame(rows)

    eligible = thr_df[(thr_df["precision"] >= 0.15) & (thr_df["recall"] >= 0.10)].copy()
    if eligible.empty:
        eligible = thr_df[thr_df["recall"] >= 0.10].copy()
    if eligible.empty:
        eligible = thr_df.copy()

    best = eligible.sort_values(["f1", "precision", "recall"], ascending=False).iloc[0]
    return float(best["threshold"])

def score_all(y_true, pred):
    return {
        "accuracy": round(accuracy_score(y_true, pred), 4),
        "precision": round(precision_score(y_true, pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, pred, zero_division=0), 4)
    }

ros = RandomOverSampler(sampling_strategy=0.45, random_state=42)

In [ ]:
# =========================
# THRESHOLD GUESS-STYLE MODEL
# =========================

tg_X_train, tg_X_val = build_threshold_guess_features(
    X_train_df, X_val_df, numeric_features, ["PRECINCT", "BORO"]
)
_, tg_X_test = build_threshold_guess_features(
    X_train_df, X_test_df, numeric_features, ["PRECINCT", "BORO"]
)

tg_X_train_ros, y_train_tg_ros = ros.fit_resample(tg_X_train, y_train)

tg_best_bundle = None

for params in ParameterGrid({
    "C": [0.05, 0.1, 0.3],
    "class_weight": ["balanced"]
}):
    model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=3000,
        random_state=42,
        **params
    )
    model.fit(tg_X_train_ros, y_train_tg_ros)

    val_proba = model.predict_proba(tg_X_val)[:, 1]
    threshold = choose_threshold(y_val, val_proba)

    val_pred = (val_proba >= threshold).astype(int)
    val_scores = score_all(y_val, val_pred)

    if (tg_best_bundle is None) or (val_scores["f1"] > tg_best_bundle["val_f1"]) or (
        val_scores["f1"] == tg_best_bundle["val_f1"] and val_scores["precision"] > tg_best_bundle["val_precision"]
    ):
        tg_best_bundle = {
            "model": model,
            "params": params,
            "threshold": threshold,
            "val_f1": val_scores["f1"],
            "val_precision": val_scores["precision"]
        }

tg_model = tg_best_bundle["model"]
tg_threshold = tg_best_bundle["threshold"]

tg_train_pred = (tg_model.predict_proba(tg_X_train)[:, 1] >= tg_threshold).astype(int)
tg_val_pred = (tg_model.predict_proba(tg_X_val)[:, 1] >= tg_threshold).astype(int)
tg_test_pred = (tg_model.predict_proba(tg_X_test)[:, 1] >= tg_threshold).astype(int)

tg_train_scores = score_all(y_train, tg_train_pred)
tg_val_scores = score_all(y_val, tg_val_pred)
tg_test_scores = score_all(y_test, tg_test_pred)

print("Threshold Guess-style")
print("Best params:", tg_best_bundle["params"])
print("Threshold:", round(tg_threshold, 2))
print("Train:", tg_train_scores)
print("Val:  ", tg_val_scores)
print("Test: ", tg_test_scores)
print(confusion_matrix(y_test, tg_test_pred))

Threshold Guess-style
Best params: {'C': 0.3, 'class_weight': 'balanced'}
Threshold: 0.75
Train: {'accuracy': 0.866, 'precision': 0.3584, 'recall': 0.3839, 'f1': 0.3707}
Val:   {'accuracy': 0.8807, 'precision': 0.2683, 'recall': 0.3952, 'f1': 0.3196}
Test:  {'accuracy': 0.8769, 'precision': 0.3333, 'recall': 0.4286, 'f1': 0.375}
[[1979  174]
 [ 116   87]]


In [ ]:
# =========================
# RBF SVM
# =========================

scaled_X_train, scaled_X_val = build_scaled_features(
    X_train_df, X_val_df, numeric_features, ["PRECINCT", "BORO"]
)
_, scaled_X_test = build_scaled_features(
    X_train_df, X_test_df, numeric_features, ["PRECINCT", "BORO"]
)

scaled_X_train_ros, y_train_scaled_ros = ros.fit_resample(scaled_X_train, y_train)

svm_best_bundle = None

for params in ParameterGrid({
    "C": [1, 3],
    "gamma": ["scale", 0.03],
    "class_weight": ["balanced"]
}):
    model = SVC(
        kernel="rbf",
        probability=True,
        random_state=42,
        **params
    )
    model.fit(scaled_X_train_ros, y_train_scaled_ros)

    val_proba = model.predict_proba(scaled_X_val)[:, 1]
    threshold = choose_threshold(y_val, val_proba)

    val_pred = (val_proba >= threshold).astype(int)
    val_scores = score_all(y_val, val_pred)

    if (svm_best_bundle is None) or (val_scores["f1"] > svm_best_bundle["val_f1"]) or (
        val_scores["f1"] == svm_best_bundle["val_f1"] and val_scores["precision"] > svm_best_bundle["val_precision"]
    ):
        svm_best_bundle = {
            "model": model,
            "params": params,
            "threshold": threshold,
            "val_f1": val_scores["f1"],
            "val_precision": val_scores["precision"]
        }

svm_model = svm_best_bundle["model"]
svm_threshold = svm_best_bundle["threshold"]

svm_train_pred = (svm_model.predict_proba(scaled_X_train)[:, 1] >= svm_threshold).astype(int)
svm_val_pred = (svm_model.predict_proba(scaled_X_val)[:, 1] >= svm_threshold).astype(int)
svm_test_pred = (svm_model.predict_proba(scaled_X_test)[:, 1] >= svm_threshold).astype(int)

svm_train_scores = score_all(y_train, svm_train_pred)
svm_val_scores = score_all(y_val, svm_val_pred)
svm_test_scores = score_all(y_test, svm_test_pred)

print("RBF SVM")
print("Best params:", svm_best_bundle["params"])
print("Threshold:", round(svm_threshold, 2))
print("Train:", svm_train_scores)
print("Val:  ", svm_val_scores)
print("Test: ", svm_test_scores)
print(confusion_matrix(y_test, svm_test_pred))

RBF SVM
Best params: {'C': 1, 'class_weight': 'balanced', 'gamma': 0.03}
Threshold: 0.35
Train: {'accuracy': 0.7683, 'precision': 0.2657, 'recall': 0.7107, 'f1': 0.3868}
Val:   {'accuracy': 0.7496, 'precision': 0.1648, 'recall': 0.6228, 'f1': 0.2607}
Test:  {'accuracy': 0.7585, 'precision': 0.2095, 'recall': 0.6502, 'f1': 0.3169}
[[1655  498]
 [  71  132]]


In [ ]:
# =========================
# NEURAL NETWORK (MLP)
# =========================

mlp_best_bundle = None

for params in ParameterGrid({
    "hidden_layer_sizes": [(32,), (64, 32)],
    "alpha": [0.001, 0.01],
    "learning_rate_init": [0.001, 0.003]
}):
    model = MLPClassifier(
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42,
        **params
    )
    model.fit(scaled_X_train_ros, y_train_scaled_ros)

    val_proba = model.predict_proba(scaled_X_val)[:, 1]
    threshold = choose_threshold(y_val, val_proba)

    val_pred = (val_proba >= threshold).astype(int)
    val_scores = score_all(y_val, val_pred)

    if (mlp_best_bundle is None) or (val_scores["f1"] > mlp_best_bundle["val_f1"]) or (
        val_scores["f1"] == mlp_best_bundle["val_f1"] and val_scores["precision"] > mlp_best_bundle["val_precision"]
    ):
        mlp_best_bundle = {
            "model": model,
            "params": params,
            "threshold": threshold,
            "val_f1": val_scores["f1"],
            "val_precision": val_scores["precision"]
        }

mlp_model = mlp_best_bundle["model"]
mlp_threshold = mlp_best_bundle["threshold"]

mlp_train_pred = (mlp_model.predict_proba(scaled_X_train)[:, 1] >= mlp_threshold).astype(int)
mlp_val_pred = (mlp_model.predict_proba(scaled_X_val)[:, 1] >= mlp_threshold).astype(int)
mlp_test_pred = (mlp_model.predict_proba(scaled_X_test)[:, 1] >= mlp_threshold).astype(int)

mlp_train_scores = score_all(y_train, mlp_train_pred)
mlp_val_scores = score_all(y_val, mlp_val_pred)
mlp_test_scores = score_all(y_test, mlp_test_pred)

print("Neural Network (MLP)")
print("Best params:", mlp_best_bundle["params"])
print("Threshold:", round(mlp_threshold, 2))
print("Train:", mlp_train_scores)
print("Val:  ", mlp_val_scores)
print("Test: ", mlp_test_scores)
print(confusion_matrix(y_test, mlp_test_pred))

Neural Network (MLP)
Best params: {'alpha': 0.01, 'hidden_layer_sizes': (32,), 'learning_rate_init': 0.003}
Threshold: 0.5
Train: {'accuracy': 0.8751, 'precision': 0.4295, 'recall': 0.6523, 'f1': 0.5179}
Val:   {'accuracy': 0.8277, 'precision': 0.1813, 'recall': 0.4072, 'f1': 0.2509}
Test:  {'accuracy': 0.809, 'precision': 0.2135, 'recall': 0.4532, 'f1': 0.2902}
[[1814  339]
 [ 111   92]]


In [ ]:
# =========================
# RESULTS TABLE
# =========================

results_df = pd.DataFrame([
    {
        "model": "Threshold Guess-style",
        "train_accuracy": tg_train_scores["accuracy"],
        "train_precision": tg_train_scores["precision"],
        "train_recall": tg_train_scores["recall"],
        "train_f1": tg_train_scores["f1"],
        "val_accuracy": tg_val_scores["accuracy"],
        "val_precision": tg_val_scores["precision"],
        "val_recall": tg_val_scores["recall"],
        "val_f1": tg_val_scores["f1"],
        "test_accuracy": tg_test_scores["accuracy"],
        "test_precision": tg_test_scores["precision"],
        "test_recall": tg_test_scores["recall"],
        "test_f1": tg_test_scores["f1"]
    },
    {
        "model": "RBF SVM",
        "train_accuracy": svm_train_scores["accuracy"],
        "train_precision": svm_train_scores["precision"],
        "train_recall": svm_train_scores["recall"],
        "train_f1": svm_train_scores["f1"],
        "val_accuracy": svm_val_scores["accuracy"],
        "val_precision": svm_val_scores["precision"],
        "val_recall": svm_val_scores["recall"],
        "val_f1": svm_val_scores["f1"],
        "test_accuracy": svm_test_scores["accuracy"],
        "test_precision": svm_test_scores["precision"],
        "test_recall": svm_test_scores["recall"],
        "test_f1": svm_test_scores["f1"]
    },
    {
        "model": "Neural Network (MLP)",
        "train_accuracy": mlp_train_scores["accuracy"],
        "train_precision": mlp_train_scores["precision"],
        "train_recall": mlp_train_scores["recall"],
        "train_f1": mlp_train_scores["f1"],
        "val_accuracy": mlp_val_scores["accuracy"],
        "val_precision": mlp_val_scores["precision"],
        "val_recall": mlp_val_scores["recall"],
        "val_f1": mlp_val_scores["f1"],
        "test_accuracy": mlp_test_scores["accuracy"],
        "test_precision": mlp_test_scores["precision"],
        "test_recall": mlp_test_scores["recall"],
        "test_f1": mlp_test_scores["f1"]
    }
]).sort_values(["val_f1", "test_f1", "test_precision"], ascending=False).reset_index(drop=True)

display(results_df)

,model,train_accuracy,train_precision,train_recall,train_f1,val_accuracy,val_precision,val_recall,val_f1,test_accuracy,test_precision,test_recall,test_f1
0,Threshold Guess-style,0.8660,0.3584,0.3839,0.3707,0.8807,0.2683,0.3952,0.3196,0.8769,0.3333,0.4286,0.3750
1,RBF SVM,0.7683,0.2657,0.7107,0.3868,0.7496,0.1648,0.6228,0.2607,0.7585,0.2095,0.6502,0.3169
2,Neural Network (MLP),0.8751,0.4295,0.6523,0.5179,0.8277,0.1813,0.4072,0.2509,0.8090,0.2135,0.4532,0.2902


In [ ]:
# =========================
# TOP THRESHOLD RULES
# =========================

rule_df = pd.DataFrame({
    "rule": tg_X_train.columns,
    "coef": tg_model.coef_[0]
})
rule_df["abs_coef"] = rule_df["coef"].abs()
rule_df = rule_df[rule_df["coef"] != 0].sort_values("abs_coef", ascending=False)

display(rule_df.head(20))

,rule,coef,abs_coef
40,PRECINCT_120,2.981284,2.981284
86,PRECINCT_75,2.504146,2.504146
89,PRECINCT_78,-2.422440,2.422440
35,PRECINCT_111,-2.419452,2.419452
49,PRECINCT_20,-2.103779,2.103779
22,PRECINCT_1,-2.097918,2.097918
46,PRECINCT_17,-2.095018,2.095018
72,PRECINCT_6,-2.084091,2.084091
102,BORO_STATEN ISLAND,-2.082539,2.082539
36,PRECINCT_112,-2.001983,2.001983


In [20]:
import os
import numpy as np
import pandas as pd

# -----------------------
# Config
# -----------------------
WEEK_START = "MON"   # "MON" or "SUN"
DATA_PATH_CANDIDATES = [
    "/mnt/data/ShootingVictim 2022-2024.csv",     # this environment
    "/content/ShootingVictim 2022-2024.csv",      # typical Colab upload path
]

DATA_PATH = next((p for p in DATA_PATH_CANDIDATES if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find the CSV. Upload it, then update DATA_PATH.")

# -----------------------
# Helpers
# -----------------------
def week_start_from_date(date_series: pd.Series, start: str = "MON") -> pd.Series:
    """
    Return week start date for each date in date_series.
    - start="MON": Monday-start weeks
    - start="SUN": Sunday-start weeks
    """
    d = pd.to_datetime(date_series)
    dow = d.dt.dayofweek  # Mon=0, Tue=1, ..., Sun=6

    if start.upper().startswith("MON"):
        offset = dow
    elif start.upper().startswith("SUN"):
        offset = (dow + 1) % 7  # Sun->0, Mon->1, ..., Sat->6
    else:
        raise ValueError("start must be 'MON' or 'SUN'.")

    return (d - pd.to_timedelta(offset, unit="D")).dt.normalize()

def aggregate_precinct_week(df_inc: pd.DataFrame, week_start: str) -> pd.DataFrame:
    df_inc = df_inc.copy()
    df_inc["OCCUR_DATE"] = pd.to_datetime(df_inc["OCCUR_DATE"])
    df_inc["week_start"] = week_start_from_date(df_inc["OCCUR_DATE"], start=week_start)

    agg = (df_inc
           .groupby(["PRECINCT", "week_start"], as_index=False)
           .agg(
               shootings_count=("INCIDENT_KEY", "nunique"),
               max_victims_in_incident=("victims_in_incident", "max"),
               Any_fatal_week=("incident_murder_flag", "max"),
           ))

    agg["hot_week"] = (
        (agg["shootings_count"] >= 2) |
        (agg["max_victims_in_incident"] >= 2) |
        (agg["Any_fatal_week"] == 1)
    ).astype(int)

    return agg

def make_full_panel(weekly_agg: pd.DataFrame, precincts: np.ndarray, week_start: str) -> pd.DataFrame:
    freq = "W-MON" if week_start.upper().startswith("MON") else "W-SUN"
    weeks = pd.date_range(weekly_agg["week_start"].min(), weekly_agg["week_start"].max(), freq=freq)

    panel = pd.MultiIndex.from_product([precincts, weeks], names=["PRECINCT", "week_start"]).to_frame(index=False)
    out = panel.merge(weekly_agg, on=["PRECINCT", "week_start"], how="left")

    for c in ["shootings_count", "max_victims_in_incident", "Any_fatal_week", "hot_week"]:
        out[c] = out[c].fillna(0).astype(int)

    return out

# -----------------------
# Load + verify counts
# -----------------------
df = pd.read_csv(DATA_PATH)
df["OCCUR_DATE"] = pd.to_datetime(df["OCCUR_DATE"])
precincts = np.sort(df["PRECINCT"].unique())

weekly_obs = aggregate_precinct_week(df, week_start=WEEK_START)
panel = make_full_panel(weekly_obs, precincts=precincts, week_start=WEEK_START)

print("Week start:", WEEK_START)
print("Hot precinct-weeks (observed):", int(weekly_obs["hot_week"].sum()))
print("Observed precinct-weeks with >=1 shooting:", int(len(weekly_obs)))
print("Full panel rows:", int(len(panel)))
print("Hot precinct-weeks (full panel):", int(panel["hot_week"].sum()))
print("Unique precincts:", int(panel["PRECINCT"].nunique()))
print("Unique weeks:", int(panel["week_start"].nunique()))


Week start: MON
Hot precinct-weeks (observed): 1111
Observed precinct-weeks with >=1 shooting: 2479
Full panel rows: 12008
Hot precinct-weeks (full panel): 1111
Unique precincts: 76
Unique weeks: 158


In [21]:
def build_weekly_base(df_inc: pd.DataFrame, week_start: str) -> pd.DataFrame:
    """
    Base precinct-week table (full 76 x weeks panel) with label + raw weekly signals.
    """
    df_inc = df_inc.copy()
    df_inc["OCCUR_DATE"] = pd.to_datetime(df_inc["OCCUR_DATE"])
    df_inc["week_start"] = week_start_from_date(df_inc["OCCUR_DATE"], start=week_start)

    # Precinct -> BORO mapping (static)
    precinct_boro = (df_inc.groupby("PRECINCT")["BORO"]
                     .agg(lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[0])
                     .rename("BORO")
                     .reset_index())

    # Weekly signals (includes your label components + extra predictors)
    weekly = (df_inc.groupby(["PRECINCT", "week_start"], as_index=False)
              .agg(
                  shootings_count=("INCIDENT_KEY", "nunique"),
                  total_victims=("victims_in_incident", "sum"),
                  max_victims_in_incident=("victims_in_incident", "max"),
                  fatal_incidents=("incident_murder_flag", "sum"),
                  Any_fatal_week=("incident_murder_flag", "max"),
                  multi_victim_incidents=("multi_victim_flag", "sum"),
                  mass_casualty_incidents=("mass_casualty_flag", "sum"),
              ))

    weekly["hot_week"] = (
        (weekly["shootings_count"] >= 2) |
        (weekly["max_victims_in_incident"] >= 2) |
        (weekly["Any_fatal_week"] == 1)
    ).astype(int)

    # Full panel
    precincts = np.sort(df_inc["PRECINCT"].unique())
    freq = "W-MON" if week_start.upper().startswith("MON") else "W-SUN"
    weeks = pd.date_range(weekly["week_start"].min(), weekly["week_start"].max(), freq=freq)

    panel = pd.MultiIndex.from_product([precincts, weeks], names=["PRECINCT", "week_start"]).to_frame(index=False)
    out = panel.merge(weekly, on=["PRECINCT", "week_start"], how="left")

    fill0 = [
        "shootings_count", "total_victims", "max_victims_in_incident",
        "fatal_incidents", "Any_fatal_week", "multi_victim_incidents",
        "mass_casualty_incidents", "hot_week"
    ]
    for c in fill0:
        out[c] = out[c].fillna(0).astype(int)

    out = out.merge(precinct_boro, on="PRECINCT", how="left")

    return out

def add_leakage_safe_features(panel_df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds lagged/rolling/expanding features that only use prior weeks.
    """
    d = panel_df.sort_values(["PRECINCT", "week_start"]).copy()

    # Time features (safe)
    d["year"] = d["week_start"].dt.year
    d["month"] = d["week_start"].dt.month
    d["weekofyear"] = d["week_start"].dt.isocalendar().week.astype(int)
    d["week_index"] = ((d["week_start"] - d["week_start"].min()).dt.days // 7).astype(int)

    d["is_summer"] = d["month"].isin([6, 7, 8]).astype(int)
    d["is_holiday_season"] = d["month"].isin([11, 12]).astype(int)

    # Columns to create lags/rolling sums from (all shifted by 1 for leakage safety)
    sum_cols = [
        "shootings_count", "total_victims", "fatal_incidents",
        "multi_victim_incidents", "mass_casualty_incidents",
        "Any_fatal_week", "hot_week",
    ]
    max_cols = ["max_victims_in_incident"]

    windows = [2, 4, 8, 12]

    for col in sum_cols:
        g = d.groupby("PRECINCT")[col]
        d[f"{col}_lag1"] = g.shift(1).fillna(0)
        d[f"{col}_lag2"] = g.shift(2).fillna(0)

        shifted = g.shift(1)
        for w in windows:
            d[f"{col}_roll_sum_{w}"] = (
                shifted.groupby(d["PRECINCT"])
                .rolling(w, min_periods=1).sum()
                .reset_index(level=0, drop=True)
                .fillna(0)
            )

    for col in max_cols:
        g = d.groupby("PRECINCT")[col]
        d[f"{col}_lag1"] = g.shift(1).fillna(0)

        shifted = g.shift(1)
        for w in windows:
            d[f"{col}_roll_max_{w}"] = (
                shifted.groupby(d["PRECINCT"])
                .rolling(w, min_periods=1).max()
                .reset_index(level=0, drop=True)
                .fillna(0)
            )

    # Longer-run precinct baselines (expanding mean on shifted history)
    for col in ["shootings_count", "hot_week", "fatal_incidents", "multi_victim_incidents", "total_victims"]:
        shifted = d.groupby("PRECINCT")[col].shift(1)
        d[f"precinct_expanding_mean_{col}"] = (
            shifted.groupby(d["PRECINCT"])
            .expanding(min_periods=4).mean()
            .reset_index(level=0, drop=True)
            .fillna(0)
        )

    # Ratios (use prior rolling values only)
    d["hot_rate_roll_8"] = d["hot_week_roll_sum_8"] / 8.0
    d["victims_per_shooting_roll_8"] = d["total_victims_roll_sum_8"] / (d["shootings_count_roll_sum_8"] + 1e-6)

    return d

# Build modeling table
panel_base = build_weekly_base(df, week_start=WEEK_START)
model_df = add_leakage_safe_features(panel_base)

print("Full panel rows:", len(model_df))
print("Hot weeks:", int(model_df["hot_week"].sum()))
print("Hot rate:", float(model_df["hot_week"].mean()))


Full panel rows: 12008
Hot weeks: 1111
Hot rate: 0.09252165223184544


In [25]:
# =========================================================
# NYC PRECINCT-WEEK SHOOTING HOTSPOT PROJECT
# Full Colab-ready pipeline
# =========================================================

# If needed in Colab:
# !pip -q install xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils import resample

from xgboost import XGBClassifier

RANDOM_STATE = 42

# =========================================================
# 1) LOAD DATA
# =========================================================
# In Colab, upload your CSV and use this path:
file_path = "/content/ShootingVictim 2022-2024.csv"
df = pd.read_csv(file_path)

# Basic cleaning
df = df[df["PRECINCT"].notna()].copy()
df["PRECINCT"] = pd.to_numeric(df["PRECINCT"], errors="coerce")
df = df.dropna(subset=["PRECINCT"]).copy()
df["PRECINCT"] = df["PRECINCT"].astype(int)

df["OCCUR_DATE"] = pd.to_datetime(df["OCCUR_DATE"], errors="coerce")
df = df.dropna(subset=["OCCUR_DATE"]).copy()

# Monday-start week
df["week_start"] = df["OCCUR_DATE"] - pd.to_timedelta(df["OCCUR_DATE"].dt.weekday, unit="D")
df["week_start"] = pd.to_datetime(df["week_start"]).dt.normalize()

# Make sure expected numeric columns are numeric
numeric_cols = [
    "victims_in_incident",
    "incident_murder_flag",
    "multi_victim_flag"
]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["victims_in_incident"] = df["victims_in_incident"].clip(lower=0)
df["incident_murder_flag"] = df["incident_murder_flag"].clip(lower=0, upper=1)
df["multi_victim_flag"] = np.where(df["victims_in_incident"] >= 2, 1, df["multi_victim_flag"])
df["multi_victim_flag"] = pd.to_numeric(df["multi_victim_flag"], errors="coerce").fillna(0).clip(0, 1)

# =========================================================
# 2) INCIDENT-LEVEL DATASET
#    One row per incident so shootings_count is correct
# =========================================================
incident_cols = [
    "INCIDENT_KEY",
    "OCCUR_DATE",
    "week_start",
    "BORO",
    "PRECINCT",
    "victims_in_incident",
    "incident_murder_flag",
    "multi_victim_flag"
]

inc = (
    df[incident_cols]
    .sort_values(["INCIDENT_KEY", "OCCUR_DATE"])
    .drop_duplicates(subset=["INCIDENT_KEY"])
    .copy()
)

# =========================================================
# 3) WEEKLY AGGREGATION + HOT LABEL
# =========================================================
weekly = (
    inc.groupby(["PRECINCT", "week_start"], as_index=False)
       .agg(
           shootings_count=("INCIDENT_KEY", "nunique"),
           total_victims_week=("victims_in_incident", "sum"),
           max_victims_in_incident=("victims_in_incident", "max"),
           any_fatal_week=("incident_murder_flag", "max"),
           multi_victim_incidents_week=("multi_victim_flag", "sum"),
       )
)

# Your hot week rule
weekly["hot_week"] = np.where(
    (weekly["shootings_count"] >= 2) |
    (weekly["max_victims_in_incident"] >= 2) |
    (weekly["any_fatal_week"] == 1),
    1, 0
)

# Borough map by precinct
boro_map = (
    inc.groupby("PRECINCT")["BORO"]
       .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else s.dropna().iloc[0])
       .to_dict()
)

# Full precinct x week panel
all_precincts = sorted(inc["PRECINCT"].dropna().unique())
all_weeks = pd.date_range(weekly["week_start"].min(), weekly["week_start"].max(), freq="W-MON")

panel = pd.MultiIndex.from_product(
    [all_precincts, all_weeks],
    names=["PRECINCT", "week_start"]
).to_frame(index=False)

panel = panel.merge(weekly, on=["PRECINCT", "week_start"], how="left")

fill_zero_cols = [
    "shootings_count",
    "total_victims_week",
    "max_victims_in_incident",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week"
]
panel[fill_zero_cols] = panel[fill_zero_cols].fillna(0)

panel["BORO"] = panel["PRECINCT"].map(boro_map)
panel = panel.sort_values(["PRECINCT", "week_start"]).reset_index(drop=True)

# =========================================================
# 4) FEATURE ENGINEERING
#    Only prior-week information
# =========================================================
def add_lag_features(data, group_col, col, lags=(1, 2, 3, 4)):
    for lag in lags:
        data[f"{col}_lag{lag}"] = data.groupby(group_col)[col].shift(lag)
    return data

def add_roll_features(data, group_col, col, windows=(2, 4, 8)):
    shifted = data.groupby(group_col)[col].shift(1)
    grouped = shifted.groupby(data[group_col])

    for w in windows:
        data[f"{col}_roll{w}_sum"] = (
            grouped.rolling(w, min_periods=1).sum().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_mean"] = (
            grouped.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_max"] = (
            grouped.rolling(w, min_periods=1).max().reset_index(level=0, drop=True)
        )
    return data

base_cols = [
    "shootings_count",
    "total_victims_week",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week",
]

for col in base_cols:
    panel = add_lag_features(panel, "PRECINCT", col, lags=(1, 2, 3, 4))
    panel = add_roll_features(panel, "PRECINCT", col, windows=(2, 4, 8))

# Severity ratios
for lag in [1, 2, 3, 4]:
    panel[f"victims_per_shooting_lag{lag}"] = np.where(
        panel[f"shootings_count_lag{lag}"] > 0,
        panel[f"total_victims_week_lag{lag}"] / panel[f"shootings_count_lag{lag}"],
        0
    )

for w in [2, 4, 8]:
    panel[f"victims_per_shooting_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"total_victims_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )
    panel[f"fatal_rate_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"any_fatal_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )
    panel[f"multi_victim_rate_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"multi_victim_incidents_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )

# Trend features
panel["shootings_trend_4_vs_8"] = panel["shootings_count_roll4_mean"] - panel["shootings_count_roll8_mean"]
panel["victims_trend_4_vs_8"] = panel["total_victims_week_roll4_mean"] - panel["total_victims_week_roll8_mean"]
panel["hot_trend_4_vs_8"] = panel["hot_week_roll4_mean"] - panel["hot_week_roll8_mean"]

# Weeks since last hot week
panel["weeks_since_last_hot"] = np.nan
for precinct, idx in panel.groupby("PRECINCT").groups.items():
    idx = list(idx)
    prior_hot = panel.loc[idx, "hot_week"].values

    vals = []
    seen_hot = False
    counter = np.nan

    for i in range(len(prior_hot)):
        if i == 0:
            vals.append(np.nan)
        else:
            if prior_hot[i - 1] == 1:
                counter = 0
                seen_hot = True
            elif seen_hot:
                counter += 1
            else:
                counter = np.nan
            vals.append(counter)

    panel.loc[idx, "weeks_since_last_hot"] = vals

# Calendar features
panel["month"] = panel["week_start"].dt.month
panel["quarter"] = panel["week_start"].dt.quarter
panel["week_of_year"] = panel["week_start"].dt.isocalendar().week.astype(int)
panel["week_sin"] = np.sin(2 * np.pi * panel["week_of_year"] / 52.0)
panel["week_cos"] = np.cos(2 * np.pi * panel["week_of_year"] / 52.0)

# Borough dummies as integers
panel = pd.get_dummies(panel, columns=["BORO"], prefix="boro", drop_first=True, dtype=int)

# Focused feature set
feature_cols = [
    "shootings_count_lag1", "shootings_count_lag2", "shootings_count_lag3", "shootings_count_lag4",
    "shootings_count_roll2_sum", "shootings_count_roll4_sum", "shootings_count_roll8_sum",
    "shootings_count_roll4_mean", "shootings_count_roll8_mean",

    "hot_week_lag1", "hot_week_lag2", "hot_week_lag3", "hot_week_lag4",
    "hot_week_roll2_sum", "hot_week_roll4_sum", "hot_week_roll8_sum",
    "hot_week_roll4_mean", "hot_week_roll8_mean",
    "weeks_since_last_hot",

    "total_victims_week_lag1", "total_victims_week_lag2", "total_victims_week_lag3", "total_victims_week_lag4",
    "total_victims_week_roll4_sum", "total_victims_week_roll8_sum",

    "any_fatal_week_lag1", "any_fatal_week_lag2",
    "any_fatal_week_roll4_sum", "any_fatal_week_roll8_sum",

    "multi_victim_incidents_week_lag1", "multi_victim_incidents_week_lag2",
    "multi_victim_incidents_week_roll4_sum", "multi_victim_incidents_week_roll8_sum",

    "victims_per_shooting_lag1", "victims_per_shooting_lag2",
    "victims_per_shooting_roll4", "victims_per_shooting_roll8",
    "fatal_rate_roll4", "fatal_rate_roll8",
    "multi_victim_rate_roll4", "multi_victim_rate_roll8",

    "shootings_trend_4_vs_8", "victims_trend_4_vs_8", "hot_trend_4_vs_8",

    "month", "quarter", "week_sin", "week_cos",
]

boro_cols = [c for c in panel.columns if c.startswith("boro_")]
feature_cols = feature_cols + boro_cols

model_df = panel[["PRECINCT", "week_start", "hot_week"] + feature_cols].copy()
model_df[feature_cols] = model_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# Force all features numeric
for c in feature_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0).astype(float)

# =========================================================
# 5) TIME-BASED TRAIN / VAL / TEST SPLIT
# =========================================================
unique_weeks = sorted(model_df["week_start"].unique())
n_weeks = len(unique_weeks)

train_end = int(n_weeks * 0.70)
val_end = int(n_weeks * 0.85)

train_weeks = unique_weeks[:train_end]
val_weeks = unique_weeks[train_end:val_end]
test_weeks = unique_weeks[val_end:]

train_df = model_df[model_df["week_start"].isin(train_weeks)].copy()
val_df = model_df[model_df["week_start"].isin(val_weeks)].copy()
test_df = model_df[model_df["week_start"].isin(test_weeks)].copy()

X_train = train_df[feature_cols].copy()
y_train = train_df["hot_week"].astype(int).values

X_val = val_df[feature_cols].copy()
y_val = val_df["hot_week"].astype(int).values

X_test = test_df[feature_cols].copy()
y_test = test_df["hot_week"].astype(int).values

print("Train shape:", X_train.shape, "hot rate:", round(y_train.mean(), 4))
print("Val shape:", X_val.shape, "hot rate:", round(y_val.mean(), 4))
print("Test shape:", X_test.shape, "hot rate:", round(y_test.mean(), 4))

# =========================================================
# 6) HELPERS
# =========================================================
def oversample_minority(X, y, target_ratio=0.40, random_state=42):
    X = X.reset_index(drop=True).copy()
    y = pd.Series(y).reset_index(drop=True)

    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index

    n_pos = len(pos_idx)
    n_neg = len(neg_idx)

    if n_pos == 0 or n_neg == 0:
        return X, y.values

    desired_pos = int((target_ratio * n_neg) / (1 - target_ratio))
    if desired_pos <= n_pos:
        return X, y.values

    add_n = desired_pos - n_pos
    sampled_pos_idx = resample(pos_idx, replace=True, n_samples=add_n, random_state=random_state)

    X_extra = X.loc[sampled_pos_idx]
    y_extra = y.loc[sampled_pos_idx]

    X_bal = pd.concat([X, X_extra], axis=0).reset_index(drop=True)
    y_bal = pd.concat([y, y_extra], axis=0).reset_index(drop=True)

    return X_bal, y_bal.values

def best_threshold_by_f1(y_true, probas, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.10, 0.91, 0.01)

    best_t = 0.50
    best_f1 = -1

    for t in thresholds:
        preds = (probas >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1

def evaluate(y_true, probas, threshold):
    preds = (probas >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

# =========================================================
# 7) THRESHOLD GUESS-STYLE BINARIZER
# =========================================================
class ThresholdGuessBinarizerSimple(BaseEstimator, TransformerMixin):
    def __init__(self, quantiles=(0.50, 0.70, 0.85, 0.95)):
        self.quantiles = quantiles

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.input_columns_ = X.columns.tolist()
        self.threshold_map_ = {}
        self.output_columns_ = []

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0)

            if vals.dtype == bool:
                vals = vals.astype(int)

            vals = vals.astype(float)
            unique_vals = np.unique(vals)

            if len(unique_vals) <= 1:
                thresholds = []
            elif len(unique_vals) == 2:
                thresholds = [float(unique_vals.max())]
            else:
                qs = np.quantile(vals, self.quantiles)
                thresholds = sorted(set(float(v) for v in qs))

            self.threshold_map_[col] = thresholds

            for t in thresholds:
                self.output_columns_.append(f"{col}__ge__{round(t, 4)}")

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = pd.DataFrame(index=X.index)

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0)

            if vals.dtype == bool:
                vals = vals.astype(int)

            vals = vals.astype(float)

            for t in self.threshold_map_[col]:
                out[f"{col}__ge__{round(t, 4)}"] = (vals >= t).astype(int)

        # in case every feature was constant
        if out.shape[1] == 0:
            out["dummy_rule"] = 0

        return out

# =========================================================
# 8) MODEL CANDIDATES
# =========================================================
pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
scale_pos_weight = neg / max(pos, 1)

model_grid = {
    "xgboost_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=150,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.90,
                colsample_bytree=0.85,
                min_child_weight=3,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                scale_pos_weight=scale_pos_weight
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=250,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.90,
                colsample_bytree=0.85,
                min_child_weight=5,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                scale_pos_weight=scale_pos_weight
            ))
        ]),
    ],

    "svm_rbf_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf",
                C=2.0,
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE
            ))
        ]),
    ],

    "neural_network_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(32,),
                alpha=0.001,
                max_iter=800,
                early_stopping=True,
                random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(64, 32),
                alpha=0.001,
                max_iter=900,
                early_stopping=True,
                random_state=RANDOM_STATE
            ))
        ]),
    ],

    "logistic_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=0.5,
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_STATE
            ))
        ]),
    ],

    "decision_tree_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=4,
                min_samples_leaf=20,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=5,
                min_samples_leaf=30,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),
    ],

    "threshold_guess_interpretable": [
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=0.30,
                class_weight="balanced",
                max_iter=4000,
                random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=1.00,
                class_weight="balanced",
                max_iter=4000,
                random_state=RANDOM_STATE
            ))
        ]),
    ],
}

# =========================================================
# 9) TRAIN / VALIDATE / TEST
# =========================================================
results = []
best_models = {}

for model_name, candidates in model_grid.items():
    best_val_f1 = -1
    best_threshold = 0.50
    best_pipe = None
    best_candidate_num = None

    for i, pipe in enumerate(candidates, start=1):
        if model_name == "neural_network_black_box":
            X_fit, y_fit = oversample_minority(X_train, y_train, target_ratio=0.40, random_state=RANDOM_STATE)
            pipe.fit(X_fit, y_fit)
        else:
            pipe.fit(X_train, y_train)

        val_proba = pipe.predict_proba(X_val)[:, 1]
        threshold, val_f1 = best_threshold_by_f1(y_val, val_proba)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_threshold = threshold
            best_pipe = pipe
            best_candidate_num = i

    X_trval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_trval = np.concatenate([y_train, y_val])

    if model_name == "neural_network_black_box":
        X_fit, y_fit = oversample_minority(X_trval, y_trval, target_ratio=0.40, random_state=RANDOM_STATE)
        best_pipe.fit(X_fit, y_fit)
    else:
        best_pipe.fit(X_trval, y_trval)

    test_proba = best_pipe.predict_proba(X_test)[:, 1]
    test_metrics = evaluate(y_test, test_proba, best_threshold)

    results.append({
        "model": model_name,
        "chosen_candidate": best_candidate_num,
        "val_best_f1": round(best_val_f1, 4),
        "test_accuracy": round(test_metrics["accuracy"], 4),
        "test_precision": round(test_metrics["precision"], 4),
        "test_recall": round(test_metrics["recall"], 4),
        "test_f1": round(test_metrics["f1"], 4),
        "chosen_threshold": round(best_threshold, 2),
        "tn": test_metrics["tn"],
        "fp": test_metrics["fp"],
        "fn": test_metrics["fn"],
        "tp": test_metrics["tp"],
    })

    best_models[model_name] = best_pipe

results_df = pd.DataFrame(results).sort_values(
    by=["test_f1", "test_recall", "test_precision", "test_accuracy"],
    ascending=False
).reset_index(drop=True)

print("\n=== FINAL MODEL RESULTS (sorted by test F1) ===")
print(results_df)

# =========================================================
# 10) OPTIONAL INTERPRETABILITY OUTPUTS
# =========================================================
if "logistic_semi_interpretable" in best_models:
    log_pipe = best_models["logistic_semi_interpretable"]
    log_clf = log_pipe.named_steps["clf"]

    logistic_importance = pd.DataFrame({
        "feature": feature_cols,
        "coef": log_clf.coef_[0]
    }).sort_values("coef", key=np.abs, ascending=False)

    print("\n=== TOP LOGISTIC FEATURES ===")
    print(logistic_importance.head(15))

if "decision_tree_interpretable" in best_models:
    tree_pipe = best_models["decision_tree_interpretable"]
    tree_clf = tree_pipe.named_steps["clf"]

    tree_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": tree_clf.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\n=== TOP DECISION TREE FEATURES ===")
    print(tree_importance.head(15))

if "xgboost_semi_interpretable" in best_models:
    xgb_pipe = best_models["xgboost_semi_interpretable"]
    xgb_clf = xgb_pipe.named_steps["clf"]

    xgb_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": xgb_clf.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\n=== TOP XGBOOST FEATURES ===")
    print(xgb_importance.head(15))

if "threshold_guess_interpretable" in best_models:
    tg_pipe = best_models["threshold_guess_interpretable"]
    tg_bin = tg_pipe.named_steps["binarizer"]
    tg_clf = tg_pipe.named_steps["clf"]

    tg_rules = pd.DataFrame({
        "rule": tg_bin.output_columns_,
        "coef": tg_clf.coef_[0]
    })

    tg_rules = tg_rules[tg_rules["coef"] != 0].sort_values("coef", key=np.abs, ascending=False)

    print("\n=== SELECTED THRESHOLD-GUESS RULES ===")
    print(tg_rules.head(20))

# =========================================================
# 11) SAVE OUTPUTS
# =========================================================
panel.to_csv("precinct_week_panel_with_features.csv", index=False)
results_df.to_csv("hotspot_model_results.csv", index=False)

print("\nSaved files:")

Train shape: (8360, 52) hot rate: 0.0971
Val shape: (1824, 52) hot rate: 0.0861
Test shape: (1824, 52) hot rate: 0.0779

=== FINAL MODEL RESULTS (sorted by test F1) ===
                           model  chosen_candidate  val_best_f1  \
0  threshold_guess_interpretable                 1       0.3547   
1    decision_tree_interpretable                 1       0.3456   
2     xgboost_semi_interpretable                 1       0.3753   
3    logistic_semi_interpretable                 1       0.3623   
4       neural_network_black_box                 1       0.3500   
5              svm_rbf_black_box                 1       0.3043   

   test_accuracy  test_precision  test_recall  test_f1  chosen_threshold  \
0         0.8832          0.3141       0.4225   0.3604              0.70   
1         0.8602          0.2731       0.4789   0.3478              0.67   
2         0.8734          0.2871       0.4225   0.3419              0.65   
3         0.8547          0.2607       0.4718   0.3358   

In [26]:
# =========================================================
# NYC PRECINCT-WEEK SHOOTING HOTSPOT PROJECT
# REDUCED 16-FEATURE VERSION
# =========================================================

# If needed in Colab:
# !pip -q install xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils import resample

from xgboost import XGBClassifier

RANDOM_STATE = 42

# =========================================================
# 1) LOAD DATA
# =========================================================
file_path = "/content/ShootingVictim 2022-2024.csv"
df = pd.read_csv(file_path)

df = df[df["PRECINCT"].notna()].copy()
df["PRECINCT"] = pd.to_numeric(df["PRECINCT"], errors="coerce")
df = df.dropna(subset=["PRECINCT"]).copy()
df["PRECINCT"] = df["PRECINCT"].astype(int)

df["OCCUR_DATE"] = pd.to_datetime(df["OCCUR_DATE"], errors="coerce")
df = df.dropna(subset=["OCCUR_DATE"]).copy()

df["week_start"] = df["OCCUR_DATE"] - pd.to_timedelta(df["OCCUR_DATE"].dt.weekday, unit="D")
df["week_start"] = pd.to_datetime(df["week_start"]).dt.normalize()

numeric_cols = [
    "victims_in_incident",
    "incident_murder_flag",
    "multi_victim_flag"
]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["victims_in_incident"] = df["victims_in_incident"].clip(lower=0)
df["incident_murder_flag"] = df["incident_murder_flag"].clip(lower=0, upper=1)
df["multi_victim_flag"] = np.where(df["victims_in_incident"] >= 2, 1, df["multi_victim_flag"])
df["multi_victim_flag"] = pd.to_numeric(df["multi_victim_flag"], errors="coerce").fillna(0).clip(0, 1)

# =========================================================
# 2) INCIDENT-LEVEL DATASET
# =========================================================
incident_cols = [
    "INCIDENT_KEY",
    "OCCUR_DATE",
    "week_start",
    "BORO",
    "PRECINCT",
    "victims_in_incident",
    "incident_murder_flag",
    "multi_victim_flag"
]

inc = (
    df[incident_cols]
    .sort_values(["INCIDENT_KEY", "OCCUR_DATE"])
    .drop_duplicates(subset=["INCIDENT_KEY"])
    .copy()
)

# =========================================================
# 3) WEEKLY AGGREGATION + HOT LABEL
# =========================================================
weekly = (
    inc.groupby(["PRECINCT", "week_start"], as_index=False)
       .agg(
           shootings_count=("INCIDENT_KEY", "nunique"),
           total_victims_week=("victims_in_incident", "sum"),
           max_victims_in_incident=("victims_in_incident", "max"),
           any_fatal_week=("incident_murder_flag", "max"),
           multi_victim_incidents_week=("multi_victim_flag", "sum"),
       )
)

weekly["hot_week"] = np.where(
    (weekly["shootings_count"] >= 2) |
    (weekly["max_victims_in_incident"] >= 2) |
    (weekly["any_fatal_week"] == 1),
    1, 0
)

all_precincts = sorted(inc["PRECINCT"].dropna().unique())
all_weeks = pd.date_range(weekly["week_start"].min(), weekly["week_start"].max(), freq="W-MON")

panel = pd.MultiIndex.from_product(
    [all_precincts, all_weeks],
    names=["PRECINCT", "week_start"]
).to_frame(index=False)

panel = panel.merge(weekly, on=["PRECINCT", "week_start"], how="left")

fill_zero_cols = [
    "shootings_count",
    "total_victims_week",
    "max_victims_in_incident",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week"
]
panel[fill_zero_cols] = panel[fill_zero_cols].fillna(0)
panel = panel.sort_values(["PRECINCT", "week_start"]).reset_index(drop=True)

# =========================================================
# 4) FEATURE ENGINEERING
# =========================================================
def add_lag_features(data, group_col, col, lags=(1, 2, 3, 4)):
    for lag in lags:
        data[f"{col}_lag{lag}"] = data.groupby(group_col)[col].shift(lag)
    return data

def add_roll_features(data, group_col, col, windows=(4, 8)):
    shifted = data.groupby(group_col)[col].shift(1)
    grouped = shifted.groupby(data[group_col])

    for w in windows:
        data[f"{col}_roll{w}_sum"] = (
            grouped.rolling(w, min_periods=1).sum().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_mean"] = (
            grouped.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        )
    return data

base_cols = [
    "shootings_count",
    "total_victims_week",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week",
]

for col in base_cols:
    panel = add_lag_features(panel, "PRECINCT", col, lags=(1, 2, 3, 4))
    panel = add_roll_features(panel, "PRECINCT", col, windows=(4, 8))

panel["victims_per_shooting_roll4"] = np.where(
    panel["shootings_count_roll4_sum"] > 0,
    panel["total_victims_week_roll4_sum"] / panel["shootings_count_roll4_sum"],
    0
)

panel["fatal_rate_roll4"] = np.where(
    panel["shootings_count_roll4_sum"] > 0,
    panel["any_fatal_week_roll4_sum"] / panel["shootings_count_roll4_sum"],
    0
)

panel["multi_victim_rate_roll4"] = np.where(
    panel["shootings_count_roll4_sum"] > 0,
    panel["multi_victim_incidents_week_roll4_sum"] / panel["shootings_count_roll4_sum"],
    0
)

panel["weeks_since_last_hot"] = np.nan
for precinct, idx in panel.groupby("PRECINCT").groups.items():
    idx = list(idx)
    prior_hot = panel.loc[idx, "hot_week"].values

    vals = []
    seen_hot = False
    counter = np.nan

    for i in range(len(prior_hot)):
        if i == 0:
            vals.append(np.nan)
        else:
            if prior_hot[i - 1] == 1:
                counter = 0
                seen_hot = True
            elif seen_hot:
                counter += 1
            else:
                counter = np.nan
            vals.append(counter)

    panel.loc[idx, "weeks_since_last_hot"] = vals

# =========================================================
# 5) REDUCED FEATURE SET (16 features)
# =========================================================
feature_cols = [
    "shootings_count_lag1",
    "shootings_count_lag2",
    "shootings_count_roll4_sum",
    "shootings_count_roll8_sum",

    "hot_week_lag1",
    "hot_week_roll4_sum",
    "weeks_since_last_hot",

    "total_victims_week_lag1",
    "total_victims_week_roll4_sum",
    "any_fatal_week_lag1",
    "any_fatal_week_roll4_sum",
    "multi_victim_incidents_week_lag1",
    "multi_victim_incidents_week_roll4_sum",

    "victims_per_shooting_roll4",
    "fatal_rate_roll4",
    "multi_victim_rate_roll4",
]

model_df = panel[["PRECINCT", "week_start", "hot_week"] + feature_cols].copy()
model_df[feature_cols] = model_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

for c in feature_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0).astype(float)

print("Number of features:", len(feature_cols))
print(feature_cols)

# =========================================================
# 6) TIME-BASED TRAIN / VAL / TEST SPLIT
# =========================================================
unique_weeks = sorted(model_df["week_start"].unique())
n_weeks = len(unique_weeks)

train_end = int(n_weeks * 0.70)
val_end = int(n_weeks * 0.85)

train_weeks = unique_weeks[:train_end]
val_weeks = unique_weeks[train_end:val_end]
test_weeks = unique_weeks[val_end:]

train_df = model_df[model_df["week_start"].isin(train_weeks)].copy()
val_df = model_df[model_df["week_start"].isin(val_weeks)].copy()
test_df = model_df[model_df["week_start"].isin(test_weeks)].copy()

X_train = train_df[feature_cols].copy()
y_train = train_df["hot_week"].astype(int).values

X_val = val_df[feature_cols].copy()
y_val = val_df["hot_week"].astype(int).values

X_test = test_df[feature_cols].copy()
y_test = test_df["hot_week"].astype(int).values

print("Train shape:", X_train.shape, "hot rate:", round(y_train.mean(), 4))
print("Val shape:", X_val.shape, "hot rate:", round(y_val.mean(), 4))
print("Test shape:", X_test.shape, "hot rate:", round(y_test.mean(), 4))

# =========================================================
# 7) HELPERS
# =========================================================
def oversample_minority(X, y, target_ratio=0.40, random_state=42):
    X = X.reset_index(drop=True).copy()
    y = pd.Series(y).reset_index(drop=True)

    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index

    n_pos = len(pos_idx)
    n_neg = len(neg_idx)

    if n_pos == 0 or n_neg == 0:
        return X, y.values

    desired_pos = int((target_ratio * n_neg) / (1 - target_ratio))
    if desired_pos <= n_pos:
        return X, y.values

    add_n = desired_pos - n_pos
    sampled_pos_idx = resample(pos_idx, replace=True, n_samples=add_n, random_state=random_state)

    X_extra = X.loc[sampled_pos_idx]
    y_extra = y.loc[sampled_pos_idx]

    X_bal = pd.concat([X, X_extra], axis=0).reset_index(drop=True)
    y_bal = pd.concat([y, y_extra], axis=0).reset_index(drop=True)

    return X_bal, y_bal.values

def best_threshold_by_f1(y_true, probas, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.10, 0.91, 0.01)

    best_t = 0.50
    best_f1 = -1

    for t in thresholds:
        preds = (probas >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1

def evaluate(y_true, probas, threshold):
    preds = (probas >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

class ThresholdGuessBinarizerSimple(BaseEstimator, TransformerMixin):
    def __init__(self, quantiles=(0.50, 0.70, 0.85, 0.95)):
        self.quantiles = quantiles

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.input_columns_ = X.columns.tolist()
        self.threshold_map_ = {}
        self.output_columns_ = []

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            unique_vals = np.unique(vals)

            if len(unique_vals) <= 1:
                thresholds = []
            elif len(unique_vals) == 2:
                thresholds = [float(unique_vals.max())]
            else:
                qs = np.quantile(vals, self.quantiles)
                thresholds = sorted(set(float(v) for v in qs))

            self.threshold_map_[col] = thresholds

            for t in thresholds:
                self.output_columns_.append(f"{col}__ge__{round(t, 4)}")

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = pd.DataFrame(index=X.index)

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            for t in self.threshold_map_[col]:
                out[f"{col}__ge__{round(t, 4)}"] = (vals >= t).astype(int)

        if out.shape[1] == 0:
            out["dummy_rule"] = 0

        return out

# =========================================================
# 8) MODELS
# =========================================================
pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
scale_pos_weight = neg / max(pos, 1)

model_grid = {
    "xgboost_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=150,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.90,
                colsample_bytree=0.85,
                min_child_weight=3,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                scale_pos_weight=scale_pos_weight
            ))
        ])
    ],

    "svm_rbf_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf",
                C=1.5,
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE
            ))
        ])
    ],

    "neural_network_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(32,),
                alpha=0.001,
                max_iter=800,
                early_stopping=True,
                random_state=RANDOM_STATE
            ))
        ])
    ],

    "logistic_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_STATE
            ))
        ])
    ],

    "decision_tree_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=4,
                min_samples_leaf=25,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ])
    ],

    "threshold_guess_interpretable": [
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=0.50,
                class_weight="balanced",
                max_iter=4000,
                random_state=RANDOM_STATE
            ))
        ])
    ],
}

# =========================================================
# 9) TRAIN / VALIDATE / TEST
# =========================================================
results = []
best_models = {}

for model_name, candidates in model_grid.items():
    best_val_f1 = -1
    best_threshold = 0.50
    best_pipe = None

    for pipe in candidates:
        if model_name == "neural_network_black_box":
            X_fit, y_fit = oversample_minority(X_train, y_train, target_ratio=0.40, random_state=RANDOM_STATE)
            pipe.fit(X_fit, y_fit)
        else:
            pipe.fit(X_train, y_train)

        val_proba = pipe.predict_proba(X_val)[:, 1]
        threshold, val_f1 = best_threshold_by_f1(y_val, val_proba)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_threshold = threshold
            best_pipe = pipe

    X_trval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_trval = np.concatenate([y_train, y_val])

    if model_name == "neural_network_black_box":
        X_fit, y_fit = oversample_minority(X_trval, y_trval, target_ratio=0.40, random_state=RANDOM_STATE)
        best_pipe.fit(X_fit, y_fit)
    else:
        best_pipe.fit(X_trval, y_trval)

    test_proba = best_pipe.predict_proba(X_test)[:, 1]
    test_metrics = evaluate(y_test, test_proba, best_threshold)

    results.append({
        "model": model_name,
        "val_best_f1": round(best_val_f1, 4),
        "test_accuracy": round(test_metrics["accuracy"], 4),
        "test_precision": round(test_metrics["precision"], 4),
        "test_recall": round(test_metrics["recall"], 4),
        "test_f1": round(test_metrics["f1"], 4),
        "chosen_threshold": round(best_threshold, 2),
        "tn": test_metrics["tn"],
        "fp": test_metrics["fp"],
        "fn": test_metrics["fn"],
        "tp": test_metrics["tp"],
    })

    best_models[model_name] = best_pipe

results_df = pd.DataFrame(results).sort_values(
    by=["test_f1", "test_recall", "test_precision", "test_accuracy"],
    ascending=False
).reset_index(drop=True)

print("\n=== FINAL MODEL RESULTS (sorted by test F1) ===")
print(results_df)

Number of features: 16
['shootings_count_lag1', 'shootings_count_lag2', 'shootings_count_roll4_sum', 'shootings_count_roll8_sum', 'hot_week_lag1', 'hot_week_roll4_sum', 'weeks_since_last_hot', 'total_victims_week_lag1', 'total_victims_week_roll4_sum', 'any_fatal_week_lag1', 'any_fatal_week_roll4_sum', 'multi_victim_incidents_week_lag1', 'multi_victim_incidents_week_roll4_sum', 'victims_per_shooting_roll4', 'fatal_rate_roll4', 'multi_victim_rate_roll4']
Train shape: (8360, 16) hot rate: 0.0971
Val shape: (1824, 16) hot rate: 0.0861
Test shape: (1824, 16) hot rate: 0.0779

=== FINAL MODEL RESULTS (sorted by test F1) ===
                           model  val_best_f1  test_accuracy  test_precision  \
0    decision_tree_interpretable       0.3486         0.8690          0.2844   
1    logistic_semi_interpretable       0.3654         0.8832          0.3081   
2     xgboost_semi_interpretable       0.3513         0.8635          0.2762   
3       neural_network_black_box       0.3626         

In [29]:
# =========================================================
# NYC PRECINCT-WEEK SHOOTING HOTSPOT PROJECT
# REDUCED FEATURE VERSION (~14 features)
# Goal: improve F1 by cutting weaker/noisier variables
# =========================================================

# If needed in Colab:
# !pip -q install xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils import resample

from xgboost import XGBClassifier

RANDOM_STATE = 42

# =========================================================
# 1) LOAD DATA
# =========================================================
file_path = "/content/ShootingVictim 2022-2024.csv"
df = pd.read_csv(file_path)

df = df[df["PRECINCT"].notna()].copy()
df["PRECINCT"] = pd.to_numeric(df["PRECINCT"], errors="coerce")
df = df.dropna(subset=["PRECINCT"]).copy()
df["PRECINCT"] = df["PRECINCT"].astype(int)

df["OCCUR_DATE"] = pd.to_datetime(df["OCCUR_DATE"], errors="coerce")
df = df.dropna(subset=["OCCUR_DATE"]).copy()

# Monday-start week
df["week_start"] = df["OCCUR_DATE"] - pd.to_timedelta(df["OCCUR_DATE"].dt.weekday, unit="D")
df["week_start"] = pd.to_datetime(df["week_start"]).dt.normalize()

numeric_cols = ["victims_in_incident", "incident_murder_flag", "multi_victim_flag"]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["victims_in_incident"] = df["victims_in_incident"].clip(lower=0)
df["incident_murder_flag"] = df["incident_murder_flag"].clip(lower=0, upper=1)
df["multi_victim_flag"] = np.where(df["victims_in_incident"] >= 2, 1, df["multi_victim_flag"])
df["multi_victim_flag"] = pd.to_numeric(df["multi_victim_flag"], errors="coerce").fillna(0).clip(0, 1)

# =========================================================
# 2) INCIDENT-LEVEL DATASET
# =========================================================
incident_cols = [
    "INCIDENT_KEY",
    "OCCUR_DATE",
    "week_start",
    "BORO",
    "PRECINCT",
    "victims_in_incident",
    "incident_murder_flag",
    "multi_victim_flag"
]

inc = (
    df[incident_cols]
    .sort_values(["INCIDENT_KEY", "OCCUR_DATE"])
    .drop_duplicates(subset=["INCIDENT_KEY"])
    .copy()
)

# =========================================================
# 3) WEEKLY AGGREGATION + HOT LABEL
# =========================================================
weekly = (
    inc.groupby(["PRECINCT", "week_start"], as_index=False)
       .agg(
           shootings_count=("INCIDENT_KEY", "nunique"),
           total_victims_week=("victims_in_incident", "sum"),
           max_victims_in_incident=("victims_in_incident", "max"),
           any_fatal_week=("incident_murder_flag", "max"),
           multi_victim_incidents_week=("multi_victim_flag", "sum"),
       )
)

weekly["hot_week"] = np.where(
    (weekly["shootings_count"] >= 2) |
    (weekly["max_victims_in_incident"] >= 2) |
    (weekly["any_fatal_week"] == 1),
    1, 0
)

all_precincts = sorted(inc["PRECINCT"].dropna().unique())
all_weeks = pd.date_range(weekly["week_start"].min(), weekly["week_start"].max(), freq="W-MON")

panel = pd.MultiIndex.from_product(
    [all_precincts, all_weeks],
    names=["PRECINCT", "week_start"]
).to_frame(index=False)

panel = panel.merge(weekly, on=["PRECINCT", "week_start"], how="left")

fill_zero_cols = [
    "shootings_count",
    "total_victims_week",
    "max_victims_in_incident",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week"
]
panel[fill_zero_cols] = panel[fill_zero_cols].fillna(0)
panel = panel.sort_values(["PRECINCT", "week_start"]).reset_index(drop=True)

# =========================================================
# 4) FEATURE ENGINEERING
# =========================================================
def add_lag_features(data, group_col, col, lags=(1, 2)):
    for lag in lags:
        data[f"{col}_lag{lag}"] = data.groupby(group_col)[col].shift(lag)
    return data

def add_roll_features(data, group_col, col, windows=(4, 8)):
    shifted = data.groupby(group_col)[col].shift(1)
    grouped = shifted.groupby(data[group_col])

    for w in windows:
        data[f"{col}_roll{w}_sum"] = (
            grouped.rolling(w, min_periods=1).sum().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_mean"] = (
            grouped.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_max"] = (
            grouped.rolling(w, min_periods=1).max().reset_index(level=0, drop=True)
        )
    return data

base_cols = [
    "shootings_count",
    "total_victims_week",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week",
]

for col in base_cols:
    panel = add_lag_features(panel, "PRECINCT", col, lags=(1, 2))
    panel = add_roll_features(panel, "PRECINCT", col, windows=(4, 8))

panel["victims_per_shooting_roll4"] = np.where(
    panel["shootings_count_roll4_sum"] > 0,
    panel["total_victims_week_roll4_sum"] / panel["shootings_count_roll4_sum"],
    0
)

panel["fatal_rate_roll4"] = np.where(
    panel["shootings_count_roll4_sum"] > 0,
    panel["any_fatal_week_roll4_sum"] / panel["shootings_count_roll4_sum"],
    0
)

panel["multi_victim_rate_roll4"] = np.where(
    panel["shootings_count_roll4_sum"] > 0,
    panel["multi_victim_incidents_week_roll4_sum"] / panel["shootings_count_roll4_sum"],
    0
)

panel["weeks_since_last_hot"] = np.nan
for precinct, idx in panel.groupby("PRECINCT").groups.items():
    idx = list(idx)
    prior_hot = panel.loc[idx, "hot_week"].values

    vals = []
    seen_hot = False
    counter = np.nan

    for i in range(len(prior_hot)):
        if i == 0:
            vals.append(np.nan)
        else:
            if prior_hot[i - 1] == 1:
                counter = 0
                seen_hot = True
            elif seen_hot:
                counter += 1
            else:
                counter = np.nan
            vals.append(counter)

    panel.loc[idx, "weeks_since_last_hot"] = vals

panel["consecutive_cold_weeks"] = np.nan
for precinct, idx in panel.groupby("PRECINCT").groups.items():
    idx = list(idx)
    hot_vals = panel.loc[idx, "hot_week"].values

    vals = []
    streak = 0
    for i in range(len(hot_vals)):
        if i == 0:
            vals.append(np.nan)
        else:
            if hot_vals[i - 1] == 0:
                streak += 1
            else:
                streak = 0
            vals.append(streak)

    panel.loc[idx, "consecutive_cold_weeks"] = vals

# =========================================================
# 5) REDUCED FEATURE SET (14 features)
# Based on your latest results
# =========================================================
feature_cols = [
    "shootings_count_lag2",
    "shootings_count_roll4_sum",
    "shootings_count_roll8_sum",
    "shootings_count_roll8_mean",

    "hot_week_lag1",
    "hot_week_lag2",
    "hot_week_roll8_sum",
    "weeks_since_last_hot",
    "consecutive_cold_weeks",

    "total_victims_week_roll4_sum",
    "any_fatal_week_roll4_sum",
    "multi_victim_incidents_week_lag1",
    "multi_victim_incidents_week_roll4_sum",
    "victims_per_shooting_roll4",
]

model_df = panel[["PRECINCT", "week_start", "hot_week"] + feature_cols].copy()
model_df[feature_cols] = model_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

for c in feature_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0).astype(float)

print("Number of features:", len(feature_cols))
print(feature_cols)

# =========================================================
# 6) TIME-BASED TRAIN / VAL / TEST SPLIT
# =========================================================
unique_weeks = sorted(model_df["week_start"].unique())
n_weeks = len(unique_weeks)

train_end = int(n_weeks * 0.70)
val_end = int(n_weeks * 0.85)

train_weeks = unique_weeks[:train_end]
val_weeks = unique_weeks[train_end:val_end]
test_weeks = unique_weeks[val_end:]

train_df = model_df[model_df["week_start"].isin(train_weeks)].copy()
val_df = model_df[model_df["week_start"].isin(val_weeks)].copy()
test_df = model_df[model_df["week_start"].isin(test_weeks)].copy()

X_train = train_df[feature_cols].copy()
y_train = train_df["hot_week"].astype(int).values

X_val = val_df[feature_cols].copy()
y_val = val_df["hot_week"].astype(int).values

X_test = test_df[feature_cols].copy()
y_test = test_df["hot_week"].astype(int).values

print("Train shape:", X_train.shape, "hot rate:", round(y_train.mean(), 4))
print("Val shape:", X_val.shape, "hot rate:", round(y_val.mean(), 4))
print("Test shape:", X_test.shape, "hot rate:", round(y_test.mean(), 4))

# =========================================================
# 7) HELPERS
# =========================================================
def oversample_minority(X, y, target_ratio=0.30, random_state=42):
    X = X.reset_index(drop=True).copy()
    y = pd.Series(y).reset_index(drop=True)

    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index

    n_pos = len(pos_idx)
    n_neg = len(neg_idx)

    if n_pos == 0 or n_neg == 0:
        return X, y.values

    desired_pos = int((target_ratio * n_neg) / (1 - target_ratio))
    if desired_pos <= n_pos:
        return X, y.values

    add_n = desired_pos - n_pos
    sampled_pos_idx = resample(pos_idx, replace=True, n_samples=add_n, random_state=random_state)

    X_extra = X.loc[sampled_pos_idx]
    y_extra = y.loc[sampled_pos_idx]

    X_bal = pd.concat([X, X_extra], axis=0).reset_index(drop=True)
    y_bal = pd.concat([y, y_extra], axis=0).reset_index(drop=True)

    return X_bal, y_bal.values

def best_threshold_by_f1(y_true, probas, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.951, 0.005)

    best_t = 0.50
    best_f1 = -1

    for t in thresholds:
        preds = (probas >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1

def evaluate(y_true, probas, threshold):
    preds = (probas >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

class ThresholdGuessBinarizerSimple(BaseEstimator, TransformerMixin):
    def __init__(self, quantiles=(0.50, 0.70, 0.85, 0.95)):
        self.quantiles = quantiles

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.input_columns_ = X.columns.tolist()
        self.threshold_map_ = {}
        self.output_columns_ = []

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            unique_vals = np.unique(vals)

            if len(unique_vals) <= 1:
                thresholds = []
            elif len(unique_vals) == 2:
                thresholds = [float(unique_vals.max())]
            else:
                qs = np.quantile(vals, self.quantiles)
                thresholds = sorted(set(float(v) for v in qs))

            self.threshold_map_[col] = thresholds

            for t in thresholds:
                self.output_columns_.append(f"{col}__ge__{round(t, 4)}")

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = pd.DataFrame(index=X.index)

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            for t in self.threshold_map_[col]:
                out[f"{col}__ge__{round(t, 4)}"] = (vals >= t).astype(int)

        if out.shape[1] == 0:
            out["dummy_rule"] = 0

        return out

# =========================================================
# 8) MODELS
# =========================================================
pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
scale_pos_weight = neg / max(pos, 1)

model_grid = {
    "xgboost_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=200,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.90,
                colsample_bytree=0.85,
                min_child_weight=3,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                scale_pos_weight=scale_pos_weight
            ))
        ])
    ],

    "svm_rbf_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf",
                C=1.5,
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_STATE
            ))
        ])
    ],

    "neural_network_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(32,),
                alpha=0.001,
                max_iter=900,
                early_stopping=True,
                random_state=RANDOM_STATE
            ))
        ])
    ],

    "logistic_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=0.7,
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_STATE
            ))
        ])
    ],

    "decision_tree_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=4,
                min_samples_leaf=25,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=5,
                min_samples_leaf=30,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),
    ],

    "threshold_guess_interpretable": [
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=0.50,
                class_weight="balanced",
                max_iter=4000,
                random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=1.00,
                class_weight="balanced",
                max_iter=4000,
                random_state=RANDOM_STATE
            ))
        ]),
    ],
}

# =========================================================
# 9) TRAIN / VALIDATE / TEST
# =========================================================
results = []
best_models = {}

for model_name, candidates in model_grid.items():
    best_val_f1 = -1
    best_threshold = 0.50
    best_pipe = None
    best_candidate_num = None

    for i, pipe in enumerate(candidates, start=1):
        if model_name == "neural_network_black_box":
            X_fit, y_fit = oversample_minority(X_train, y_train, target_ratio=0.30, random_state=RANDOM_STATE)
            pipe.fit(X_fit, y_fit)
        else:
            pipe.fit(X_train, y_train)

        val_proba = pipe.predict_proba(X_val)[:, 1]
        threshold, val_f1 = best_threshold_by_f1(y_val, val_proba)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_threshold = threshold
            best_pipe = pipe
            best_candidate_num = i

    X_trval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_trval = np.concatenate([y_train, y_val])

    if model_name == "neural_network_black_box":
        X_fit, y_fit = oversample_minority(X_trval, y_trval, target_ratio=0.30, random_state=RANDOM_STATE)
        best_pipe.fit(X_fit, y_fit)
    else:
        best_pipe.fit(X_trval, y_trval)

    test_proba = best_pipe.predict_proba(X_test)[:, 1]
    test_metrics = evaluate(y_test, test_proba, best_threshold)

    results.append({
        "model": model_name,
        "chosen_candidate": best_candidate_num,
        "val_best_f1": round(best_val_f1, 4),
        "test_accuracy": round(test_metrics["accuracy"], 4),
        "test_precision": round(test_metrics["precision"], 4),
        "test_recall": round(test_metrics["recall"], 4),
        "test_f1": round(test_metrics["f1"], 4),
        "chosen_threshold": round(best_threshold, 3),
        "tn": test_metrics["tn"],
        "fp": test_metrics["fp"],
        "fn": test_metrics["fn"],
        "tp": test_metrics["tp"],
    })

    best_models[model_name] = best_pipe

results_df = pd.DataFrame(results).sort_values(
    by=["test_f1", "test_recall", "test_precision", "test_accuracy"],
    ascending=False
).reset_index(drop=True)

print("\n=== FINAL MODEL RESULTS (sorted by test F1) ===")
print(results_df)

# =========================================================
# 10) OPTIONAL INTERPRETABILITY OUTPUTS
# =========================================================
if "logistic_semi_interpretable" in best_models:
    log_pipe = best_models["logistic_semi_interpretable"]
    log_clf = log_pipe.named_steps["clf"]

    logistic_importance = pd.DataFrame({
        "feature": feature_cols,
        "coef": log_clf.coef_[0]
    }).sort_values("coef", key=np.abs, ascending=False)

    print("\n=== TOP LOGISTIC FEATURES ===")
    print(logistic_importance)

if "decision_tree_interpretable" in best_models:
    tree_pipe = best_models["decision_tree_interpretable"]
    tree_clf = tree_pipe.named_steps["clf"]

    tree_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": tree_clf.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\n=== TOP DECISION TREE FEATURES ===")
    print(tree_importance)

if "xgboost_semi_interpretable" in best_models:
    xgb_pipe = best_models["xgboost_semi_interpretable"]
    xgb_clf = xgb_pipe.named_steps["clf"]

    xgb_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": xgb_clf.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\n=== TOP XGBOOST FEATURES ===")
    print(xgb_importance)

# =========================================================
# 11) SAVE OUTPUTS
# =========================================================
panel.to_csv("precinct_week_panel_with_features_reduced.csv", index=False)
results_df.to_csv("hotspot_model_results_reduced.csv", index=False)

print("\nSaved files:")
print("- precinct_week_panel_with_features_reduced.csv")
print("- hotspot_model_results_reduced.csv")

Number of features: 14
['shootings_count_lag2', 'shootings_count_roll4_sum', 'shootings_count_roll8_sum', 'shootings_count_roll8_mean', 'hot_week_lag1', 'hot_week_lag2', 'hot_week_roll8_sum', 'weeks_since_last_hot', 'consecutive_cold_weeks', 'total_victims_week_roll4_sum', 'any_fatal_week_roll4_sum', 'multi_victim_incidents_week_lag1', 'multi_victim_incidents_week_roll4_sum', 'victims_per_shooting_roll4']
Train shape: (8360, 14) hot rate: 0.0971
Val shape: (1824, 14) hot rate: 0.0861
Test shape: (1824, 14) hot rate: 0.0779

=== FINAL MODEL RESULTS (sorted by test F1) ===
                           model  chosen_candidate  val_best_f1  \
0    logistic_semi_interpretable                 1       0.3671   
1     xgboost_semi_interpretable                 1       0.3605   
2    decision_tree_interpretable                 1       0.3516   
3       neural_network_black_box                 1       0.3631   
4  threshold_guess_interpretable                 2       0.3725   
5              svm_r

In [30]:
# =========================================================
# NYC PRECINCT-WEEK SHOOTING HOTSPOT PROJECT
# CORE 8 FEATURES + ALL 6 MODELS
# Prints only the important tuning information
# No CSV outputs
# =========================================================

# If needed in Colab:
# !pip -q install xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils import resample
from xgboost import XGBClassifier

RANDOM_STATE = 42

# =========================================================
# 1) LOAD DATA
# =========================================================
file_path = "/content/ShootingVictim 2022-2024.csv"
df = pd.read_csv(file_path)

df = df[df["PRECINCT"].notna()].copy()
df["PRECINCT"] = pd.to_numeric(df["PRECINCT"], errors="coerce")
df = df.dropna(subset=["PRECINCT"]).copy()
df["PRECINCT"] = df["PRECINCT"].astype(int)

df["OCCUR_DATE"] = pd.to_datetime(df["OCCUR_DATE"], errors="coerce")
df = df.dropna(subset=["OCCUR_DATE"]).copy()

df["week_start"] = df["OCCUR_DATE"] - pd.to_timedelta(df["OCCUR_DATE"].dt.weekday, unit="D")
df["week_start"] = pd.to_datetime(df["week_start"]).dt.normalize()

numeric_cols = ["victims_in_incident", "incident_murder_flag", "multi_victim_flag"]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["victims_in_incident"] = df["victims_in_incident"].clip(lower=0)
df["incident_murder_flag"] = df["incident_murder_flag"].clip(lower=0, upper=1)
df["multi_victim_flag"] = np.where(df["victims_in_incident"] >= 2, 1, df["multi_victim_flag"])
df["multi_victim_flag"] = pd.to_numeric(df["multi_victim_flag"], errors="coerce").fillna(0).clip(0, 1)

# =========================================================
# 2) INCIDENT LEVEL
# =========================================================
incident_cols = [
    "INCIDENT_KEY",
    "OCCUR_DATE",
    "week_start",
    "PRECINCT",
    "victims_in_incident",
    "incident_murder_flag",
    "multi_victim_flag"
]

inc = (
    df[incident_cols]
    .sort_values(["INCIDENT_KEY", "OCCUR_DATE"])
    .drop_duplicates(subset=["INCIDENT_KEY"])
    .copy()
)

# =========================================================
# 3) WEEKLY AGGREGATION + HOT LABEL
# =========================================================
weekly = (
    inc.groupby(["PRECINCT", "week_start"], as_index=False)
       .agg(
           shootings_count=("INCIDENT_KEY", "nunique"),
           total_victims_week=("victims_in_incident", "sum"),
           max_victims_in_incident=("victims_in_incident", "max"),
           any_fatal_week=("incident_murder_flag", "max"),
           multi_victim_incidents_week=("multi_victim_flag", "sum"),
       )
)

weekly["hot_week"] = np.where(
    (weekly["shootings_count"] >= 2) |
    (weekly["max_victims_in_incident"] >= 2) |
    (weekly["any_fatal_week"] == 1),
    1, 0
)

all_precincts = sorted(inc["PRECINCT"].dropna().unique())
all_weeks = pd.date_range(weekly["week_start"].min(), weekly["week_start"].max(), freq="W-MON")

panel = pd.MultiIndex.from_product(
    [all_precincts, all_weeks],
    names=["PRECINCT", "week_start"]
).to_frame(index=False)

panel = panel.merge(weekly, on=["PRECINCT", "week_start"], how="left")

fill_zero_cols = [
    "shootings_count",
    "total_victims_week",
    "max_victims_in_incident",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week"
]
panel[fill_zero_cols] = panel[fill_zero_cols].fillna(0)
panel = panel.sort_values(["PRECINCT", "week_start"]).reset_index(drop=True)

# =========================================================
# 4) FEATURE ENGINEERING
# =========================================================
def add_lag_features(data, group_col, col, lags=(1, 2)):
    for lag in lags:
        data[f"{col}_lag{lag}"] = data.groupby(group_col)[col].shift(lag)
    return data

def add_roll_features(data, group_col, col, windows=(4, 8)):
    shifted = data.groupby(group_col)[col].shift(1)
    grouped = shifted.groupby(data[group_col])

    for w in windows:
        data[f"{col}_roll{w}_sum"] = (
            grouped.rolling(w, min_periods=1).sum().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_mean"] = (
            grouped.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        )
    return data

base_cols = [
    "shootings_count",
    "total_victims_week",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week",
]

for col in base_cols:
    panel = add_lag_features(panel, "PRECINCT", col, lags=(1, 2))
    panel = add_roll_features(panel, "PRECINCT", col, windows=(4, 8))

panel["weeks_since_last_hot"] = np.nan
for precinct, idx in panel.groupby("PRECINCT").groups.items():
    idx = list(idx)
    prior_hot = panel.loc[idx, "hot_week"].values

    vals = []
    seen_hot = False
    counter = np.nan

    for i in range(len(prior_hot)):
        if i == 0:
            vals.append(np.nan)
        else:
            if prior_hot[i - 1] == 1:
                counter = 0
                seen_hot = True
            elif seen_hot:
                counter += 1
            else:
                counter = np.nan
            vals.append(counter)

    panel.loc[idx, "weeks_since_last_hot"] = vals

panel["consecutive_cold_weeks"] = np.nan
for precinct, idx in panel.groupby("PRECINCT").groups.items():
    idx = list(idx)
    hot_vals = panel.loc[idx, "hot_week"].values

    vals = []
    streak = 0
    for i in range(len(hot_vals)):
        if i == 0:
            vals.append(np.nan)
        else:
            if hot_vals[i - 1] == 0:
                streak += 1
            else:
                streak = 0
            vals.append(streak)

    panel.loc[idx, "consecutive_cold_weeks"] = vals

# =========================================================
# 5) CORE 8 FEATURES
# =========================================================
feature_cols = [
    "shootings_count_roll8_mean",
    "shootings_count_roll8_sum",
    "shootings_count_roll4_sum",
    "consecutive_cold_weeks",
    "weeks_since_last_hot",
    "hot_week_roll8_sum",
    "hot_week_lag2",
    "any_fatal_week_roll4_sum",
]

model_df = panel[["PRECINCT", "week_start", "hot_week"] + feature_cols].copy()
model_df[feature_cols] = model_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

for c in feature_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0).astype(float)

print("Number of features:", len(feature_cols))
print(feature_cols)

# =========================================================
# 6) TIME SPLIT
# =========================================================
unique_weeks = sorted(model_df["week_start"].unique())
n_weeks = len(unique_weeks)

train_end = int(n_weeks * 0.70)
val_end = int(n_weeks * 0.85)

train_weeks = unique_weeks[:train_end]
val_weeks = unique_weeks[train_end:val_end]
test_weeks = unique_weeks[val_end:]

train_df = model_df[model_df["week_start"].isin(train_weeks)].copy()
val_df = model_df[model_df["week_start"].isin(val_weeks)].copy()
test_df = model_df[model_df["week_start"].isin(test_weeks)].copy()

X_train = train_df[feature_cols].copy()
y_train = train_df["hot_week"].astype(int).values

X_val = val_df[feature_cols].copy()
y_val = val_df["hot_week"].astype(int).values

X_test = test_df[feature_cols].copy()
y_test = test_df["hot_week"].astype(int).values

print("Train shape:", X_train.shape, "hot rate:", round(y_train.mean(), 4))
print("Val shape:", X_val.shape, "hot rate:", round(y_val.mean(), 4))
print("Test shape:", X_test.shape, "hot rate:", round(y_test.mean(), 4))

# =========================================================
# 7) HELPERS
# =========================================================
def oversample_minority(X, y, target_ratio=0.30, random_state=42):
    X = X.reset_index(drop=True).copy()
    y = pd.Series(y).reset_index(drop=True)

    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index

    n_pos = len(pos_idx)
    n_neg = len(neg_idx)

    if n_pos == 0 or n_neg == 0:
        return X, y.values

    desired_pos = int((target_ratio * n_neg) / (1 - target_ratio))
    if desired_pos <= n_pos:
        return X, y.values

    add_n = desired_pos - n_pos
    sampled_pos_idx = resample(pos_idx, replace=True, n_samples=add_n, random_state=random_state)

    X_extra = X.loc[sampled_pos_idx]
    y_extra = y.loc[sampled_pos_idx]

    X_bal = pd.concat([X, X_extra], axis=0).reset_index(drop=True)
    y_bal = pd.concat([y, y_extra], axis=0).reset_index(drop=True)

    return X_bal, y_bal.values

def best_threshold_by_f1(y_true, probas, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.35, 0.851, 0.005)

    best_t = 0.50
    best_f1 = -1

    for t in thresholds:
        preds = (probas >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1

def evaluate(y_true, probas, threshold):
    preds = (probas >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

class ThresholdGuessBinarizerSimple(BaseEstimator, TransformerMixin):
    def __init__(self, quantiles=(0.50, 0.70, 0.85, 0.95)):
        self.quantiles = quantiles

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.input_columns_ = X.columns.tolist()
        self.threshold_map_ = {}
        self.output_columns_ = []

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            unique_vals = np.unique(vals)

            if len(unique_vals) <= 1:
                thresholds = []
            elif len(unique_vals) == 2:
                thresholds = [float(unique_vals.max())]
            else:
                qs = np.quantile(vals, self.quantiles)
                thresholds = sorted(set(float(v) for v in qs))

            self.threshold_map_[col] = thresholds

            for t in thresholds:
                self.output_columns_.append(f"{col}__ge__{round(t, 4)}")

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = pd.DataFrame(index=X.index)

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            for t in self.threshold_map_[col]:
                out[f"{col}__ge__{round(t, 4)}"] = (vals >= t).astype(int)

        if out.shape[1] == 0:
            out["dummy_rule"] = 0

        return out

# =========================================================
# 8) ALL 6 MODELS
# =========================================================
pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
scale_pos_weight = neg / max(pos, 1)

model_grid = {
    "logistic_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.2, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.5, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.7, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
    ],

    "xgboost_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=100, max_depth=2, learning_rate=0.05,
                subsample=0.9, colsample_bytree=0.9, min_child_weight=3,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=150, max_depth=2, learning_rate=0.05,
                subsample=0.9, colsample_bytree=0.9, min_child_weight=5,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=150, max_depth=3, learning_rate=0.03,
                subsample=0.9, colsample_bytree=0.9, min_child_weight=5,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
    ],

    "decision_tree_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=3, min_samples_leaf=30,
                class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=4, min_samples_leaf=40,
                class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=4, min_samples_leaf=50,
                class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
    ],

    "threshold_guess_interpretable": [
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=0.30,
                class_weight="balanced", max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=0.50,
                class_weight="balanced", max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=1.00,
                class_weight="balanced", max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
    ],

    "svm_rbf_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=1.0, gamma="scale",
                class_weight="balanced", probability=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=1.5, gamma="scale",
                class_weight="balanced", probability=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=2.0, gamma="scale",
                class_weight="balanced", probability=True, random_state=RANDOM_STATE
            ))
        ]),
    ],

    "neural_network_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(16,),
                alpha=0.001, max_iter=800,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(32,),
                alpha=0.001, max_iter=900,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(32, 16),
                alpha=0.001, max_iter=1000,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
    ],
}

# =========================================================
# 9) TRAIN / VALIDATE / TEST
# =========================================================
results = []
best_models = {}

for model_name, candidates in model_grid.items():
    best_val_f1 = -1
    best_threshold = 0.50
    best_pipe = None
    best_candidate_num = None

    for i, pipe in enumerate(candidates, start=1):
        if model_name == "neural_network_black_box":
            X_fit, y_fit = oversample_minority(X_train, y_train, target_ratio=0.30, random_state=RANDOM_STATE)
            pipe.fit(X_fit, y_fit)
        else:
            pipe.fit(X_train, y_train)

        val_proba = pipe.predict_proba(X_val)[:, 1]
        threshold, val_f1 = best_threshold_by_f1(y_val, val_proba)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_threshold = threshold
            best_pipe = pipe
            best_candidate_num = i

    X_trval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
    y_trval = np.concatenate([y_train, y_val])

    if model_name == "neural_network_black_box":
        X_fit, y_fit = oversample_minority(X_trval, y_trval, target_ratio=0.30, random_state=RANDOM_STATE)
        best_pipe.fit(X_fit, y_fit)
    else:
        best_pipe.fit(X_trval, y_trval)

    test_proba = best_pipe.predict_proba(X_test)[:, 1]
    test_metrics = evaluate(y_test, test_proba, best_threshold)

    results.append({
        "model": model_name,
        "chosen_candidate": best_candidate_num,
        "val_best_f1": round(best_val_f1, 4),
        "test_accuracy": round(test_metrics["accuracy"], 4),
        "test_precision": round(test_metrics["precision"], 4),
        "test_recall": round(test_metrics["recall"], 4),
        "test_f1": round(test_metrics["f1"], 4),
        "chosen_threshold": round(best_threshold, 3),
        "tn": test_metrics["tn"],
        "fp": test_metrics["fp"],
        "fn": test_metrics["fn"],
        "tp": test_metrics["tp"],
    })

    best_models[model_name] = best_pipe

results_df = pd.DataFrame(results).sort_values(
    by=["test_f1", "test_recall", "test_precision", "test_accuracy"],
    ascending=False
).reset_index(drop=True)

print("\n=== FINAL MODEL RESULTS (sorted by test F1) ===")
print(results_df)

# =========================================================
# 10) IMPORTANT PRINTS FOR FINE TUNING
# =========================================================
if "logistic_semi_interpretable" in best_models:
    log_pipe = best_models["logistic_semi_interpretable"]
    log_clf = log_pipe.named_steps["clf"]

    logistic_importance = pd.DataFrame({
        "feature": feature_cols,
        "coef": log_clf.coef_[0]
    }).sort_values("coef", key=np.abs, ascending=False)

    print("\n=== TOP LOGISTIC FEATURES ===")
    print(logistic_importance)

if "decision_tree_interpretable" in best_models:
    tree_pipe = best_models["decision_tree_interpretable"]
    tree_clf = tree_pipe.named_steps["clf"]

    tree_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": tree_clf.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\n=== TOP DECISION TREE FEATURES ===")
    print(tree_importance)

if "xgboost_semi_interpretable" in best_models:
    xgb_pipe = best_models["xgboost_semi_interpretable"]
    xgb_clf = xgb_pipe.named_steps["clf"]

    xgb_importance = pd.DataFrame({
        "feature": feature_cols,
        "importance": xgb_clf.feature_importances_
    }).sort_values("importance", ascending=False)

    print("\n=== TOP XGBOOST FEATURES ===")
    print(xgb_importance)

if "threshold_guess_interpretable" in best_models:
    tg_pipe = best_models["threshold_guess_interpretable"]
    tg_bin = tg_pipe.named_steps["binarizer"]
    tg_clf = tg_pipe.named_steps["clf"]

    tg_rules = pd.DataFrame({
        "rule": tg_bin.output_columns_,
        "coef": tg_clf.coef_[0]
    })

    tg_rules = tg_rules[tg_rules["coef"] != 0].sort_values("coef", key=np.abs, ascending=False)

    print("\n=== SELECTED THRESHOLD-GUESS RULES ===")
    print(tg_rules.head(25))

print("\n=== BEST MODEL ONLY ===")
print(results_df.iloc[0])

Number of features: 8
['shootings_count_roll8_mean', 'shootings_count_roll8_sum', 'shootings_count_roll4_sum', 'consecutive_cold_weeks', 'weeks_since_last_hot', 'hot_week_roll8_sum', 'hot_week_lag2', 'any_fatal_week_roll4_sum']
Train shape: (8360, 8) hot rate: 0.0971
Val shape: (1824, 8) hot rate: 0.0861
Test shape: (1824, 8) hot rate: 0.0779

=== FINAL MODEL RESULTS (sorted by test F1) ===
                           model  chosen_candidate  val_best_f1  \
0    logistic_semi_interpretable                 1       0.3782   
1     xgboost_semi_interpretable                 3       0.3718   
2    decision_tree_interpretable                 1       0.3506   
3       neural_network_black_box                 2       0.3630   
4  threshold_guess_interpretable                 3       0.3585   
5              svm_rbf_black_box                 1       0.0000   

   test_accuracy  test_precision  test_recall  test_f1  chosen_threshold  \
0         0.8766          0.3033       0.4507   0.3626      

In [31]:
# =========================================================
# NYC PRECINCT-WEEK SHOOTING HOTSPOT PROJECT
# STRONGER F1 SEARCH
# - tests multiple feature sets
# - keeps all 6 models
# - tunes thresholds for F1
# - prints only useful outputs
# - no files saved
# =========================================================

# If needed in Colab:
# !pip -q install xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils import resample

from xgboost import XGBClassifier

RANDOM_STATE = 42

# =========================================================
# 1) LOAD DATA
# =========================================================
file_path = "/content/ShootingVictim 2022-2024.csv"
df = pd.read_csv(file_path)

df = df[df["PRECINCT"].notna()].copy()
df["PRECINCT"] = pd.to_numeric(df["PRECINCT"], errors="coerce")
df = df.dropna(subset=["PRECINCT"]).copy()
df["PRECINCT"] = df["PRECINCT"].astype(int)

df["OCCUR_DATE"] = pd.to_datetime(df["OCCUR_DATE"], errors="coerce")
df = df.dropna(subset=["OCCUR_DATE"]).copy()

# Monday-start week
df["week_start"] = df["OCCUR_DATE"] - pd.to_timedelta(df["OCCUR_DATE"].dt.weekday, unit="D")
df["week_start"] = pd.to_datetime(df["week_start"]).dt.normalize()

numeric_cols = ["victims_in_incident", "incident_murder_flag", "multi_victim_flag"]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["victims_in_incident"] = df["victims_in_incident"].clip(lower=0)
df["incident_murder_flag"] = df["incident_murder_flag"].clip(lower=0, upper=1)
df["multi_victim_flag"] = np.where(df["victims_in_incident"] >= 2, 1, df["multi_victim_flag"])
df["multi_victim_flag"] = pd.to_numeric(df["multi_victim_flag"], errors="coerce").fillna(0).clip(0, 1)

# =========================================================
# 2) INCIDENT LEVEL
# =========================================================
incident_cols = [
    "INCIDENT_KEY",
    "OCCUR_DATE",
    "week_start",
    "PRECINCT",
    "victims_in_incident",
    "incident_murder_flag",
    "multi_victim_flag"
]

inc = (
    df[incident_cols]
    .sort_values(["INCIDENT_KEY", "OCCUR_DATE"])
    .drop_duplicates(subset=["INCIDENT_KEY"])
    .copy()
)

# =========================================================
# 3) WEEKLY AGGREGATION + HOT LABEL
# =========================================================
weekly = (
    inc.groupby(["PRECINCT", "week_start"], as_index=False)
       .agg(
           shootings_count=("INCIDENT_KEY", "nunique"),
           total_victims_week=("victims_in_incident", "sum"),
           max_victims_in_incident=("victims_in_incident", "max"),
           any_fatal_week=("incident_murder_flag", "max"),
           multi_victim_incidents_week=("multi_victim_flag", "sum"),
       )
)

weekly["hot_week"] = np.where(
    (weekly["shootings_count"] >= 2) |
    (weekly["max_victims_in_incident"] >= 2) |
    (weekly["any_fatal_week"] == 1),
    1, 0
)

all_precincts = sorted(inc["PRECINCT"].dropna().unique())
all_weeks = pd.date_range(weekly["week_start"].min(), weekly["week_start"].max(), freq="W-MON")

panel = pd.MultiIndex.from_product(
    [all_precincts, all_weeks],
    names=["PRECINCT", "week_start"]
).to_frame(index=False)

panel = panel.merge(weekly, on=["PRECINCT", "week_start"], how="left")

fill_zero_cols = [
    "shootings_count",
    "total_victims_week",
    "max_victims_in_incident",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week"
]
panel[fill_zero_cols] = panel[fill_zero_cols].fillna(0)
panel = panel.sort_values(["PRECINCT", "week_start"]).reset_index(drop=True)

# =========================================================
# 4) FEATURE ENGINEERING
# =========================================================
def add_lag_features(data, group_col, col, lags=(1, 2, 3, 4)):
    for lag in lags:
        data[f"{col}_lag{lag}"] = data.groupby(group_col)[col].shift(lag)
    return data

def add_roll_features(data, group_col, col, windows=(2, 4, 8)):
    shifted = data.groupby(group_col)[col].shift(1)
    grouped = shifted.groupby(data[group_col])

    for w in windows:
        data[f"{col}_roll{w}_sum"] = (
            grouped.rolling(w, min_periods=1).sum().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_mean"] = (
            grouped.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_max"] = (
            grouped.rolling(w, min_periods=1).max().reset_index(level=0, drop=True)
        )
    return data

base_cols = [
    "shootings_count",
    "total_victims_week",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week",
]

for col in base_cols:
    panel = add_lag_features(panel, "PRECINCT", col, lags=(1, 2, 3, 4))
    panel = add_roll_features(panel, "PRECINCT", col, windows=(2, 4, 8))

# ratios
for w in [2, 4, 8]:
    panel[f"victims_per_shooting_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"total_victims_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )
    panel[f"fatal_rate_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"any_fatal_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )
    panel[f"multi_victim_rate_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"multi_victim_incidents_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )

# memory features
panel["weeks_since_last_hot"] = np.nan
panel["consecutive_cold_weeks"] = np.nan

for precinct, idx in panel.groupby("PRECINCT").groups.items():
    idx = list(idx)
    hot_vals = panel.loc[idx, "hot_week"].values

    ws_vals = []
    cold_vals = []

    seen_hot = False
    since_hot = np.nan
    cold_streak = 0

    for i in range(len(hot_vals)):
        if i == 0:
            ws_vals.append(np.nan)
            cold_vals.append(np.nan)
            continue

        # weeks since last hot
        if hot_vals[i - 1] == 1:
            since_hot = 0
            seen_hot = True
        elif seen_hot:
            since_hot += 1
        else:
            since_hot = np.nan
        ws_vals.append(since_hot)

        # consecutive cold
        if hot_vals[i - 1] == 0:
            cold_streak += 1
        else:
            cold_streak = 0
        cold_vals.append(cold_streak)

    panel.loc[idx, "weeks_since_last_hot"] = ws_vals
    panel.loc[idx, "consecutive_cold_weeks"] = cold_vals

# interaction / trend features
panel["shooting_hot_interaction"] = panel["shootings_count_roll4_sum"] * panel["hot_week_roll4_sum"]
panel["victim_fatal_interaction"] = panel["total_victims_week_roll4_sum"] * panel["any_fatal_week_roll4_sum"]
panel["shootings_trend_4_vs_8"] = panel["shootings_count_roll4_mean"] - panel["shootings_count_roll8_mean"]
panel["victims_trend_4_vs_8"] = panel["total_victims_week_roll4_mean"] - panel["total_victims_week_roll8_mean"]
panel["hot_trend_4_vs_8"] = panel["hot_week_roll4_mean"] - panel["hot_week_roll8_mean"]

# =========================================================
# 5) FEATURE SET CANDIDATES
# =========================================================
feature_sets = {
    "core_8": [
        "shootings_count_roll8_mean",
        "shootings_count_roll8_sum",
        "shootings_count_roll4_sum",
        "consecutive_cold_weeks",
        "weeks_since_last_hot",
        "hot_week_roll8_sum",
        "hot_week_lag2",
        "any_fatal_week_roll4_sum",
    ],

    "balanced_12": [
        "shootings_count_roll8_mean",
        "shootings_count_roll8_sum",
        "shootings_count_roll4_sum",
        "shootings_count_lag2",
        "consecutive_cold_weeks",
        "weeks_since_last_hot",
        "hot_week_roll8_sum",
        "hot_week_lag1",
        "hot_week_lag2",
        "any_fatal_week_roll4_sum",
        "total_victims_week_roll4_sum",
        "victims_per_shooting_roll4",
    ],

    "strong_16": [
        "shootings_count_roll8_mean",
        "shootings_count_roll8_sum",
        "shootings_count_roll4_sum",
        "shootings_count_lag2",
        "shootings_count_roll4_max",
        "consecutive_cold_weeks",
        "weeks_since_last_hot",
        "hot_week_roll8_sum",
        "hot_week_roll4_sum",
        "hot_week_lag1",
        "hot_week_lag2",
        "any_fatal_week_roll4_sum",
        "total_victims_week_roll4_sum",
        "multi_victim_incidents_week_roll4_sum",
        "victims_per_shooting_roll4",
        "shooting_hot_interaction",
    ],

    "strong_20": [
        "shootings_count_roll8_mean",
        "shootings_count_roll8_sum",
        "shootings_count_roll4_sum",
        "shootings_count_lag2",
        "shootings_count_roll4_max",
        "consecutive_cold_weeks",
        "weeks_since_last_hot",
        "hot_week_roll8_sum",
        "hot_week_roll4_sum",
        "hot_week_lag1",
        "hot_week_lag2",
        "any_fatal_week_roll4_sum",
        "any_fatal_week_roll4_max",
        "total_victims_week_roll4_sum",
        "total_victims_week_roll4_max",
        "multi_victim_incidents_week_lag1",
        "multi_victim_incidents_week_roll4_sum",
        "victims_per_shooting_roll4",
        "fatal_rate_roll4",
        "shooting_hot_interaction",
    ],
}

# clean numeric
all_needed_features = sorted(set(sum(feature_sets.values(), [])))
model_df = panel[["PRECINCT", "week_start", "hot_week"] + all_needed_features].copy()
model_df[all_needed_features] = model_df[all_needed_features].replace([np.inf, -np.inf], np.nan).fillna(0)

for c in all_needed_features:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0).astype(float)

# =========================================================
# 6) TIME SPLIT
# =========================================================
unique_weeks = sorted(model_df["week_start"].unique())
n_weeks = len(unique_weeks)

train_end = int(n_weeks * 0.70)
val_end = int(n_weeks * 0.85)

train_weeks = unique_weeks[:train_end]
val_weeks = unique_weeks[train_end:val_end]
test_weeks = unique_weeks[val_end:]

train_df = model_df[model_df["week_start"].isin(train_weeks)].copy()
val_df = model_df[model_df["week_start"].isin(val_weeks)].copy()
test_df = model_df[model_df["week_start"].isin(test_weeks)].copy()

print("Train weeks:", len(train_weeks), "Val weeks:", len(val_weeks), "Test weeks:", len(test_weeks))
print("Train hot rate:", round(train_df["hot_week"].mean(), 4))
print("Val hot rate:", round(val_df["hot_week"].mean(), 4))
print("Test hot rate:", round(test_df["hot_week"].mean(), 4))

# =========================================================
# 7) HELPERS
# =========================================================
def oversample_minority(X, y, target_ratio=0.30, random_state=42):
    X = X.reset_index(drop=True).copy()
    y = pd.Series(y).reset_index(drop=True)

    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index

    n_pos = len(pos_idx)
    n_neg = len(neg_idx)

    if n_pos == 0 or n_neg == 0:
        return X, y.values

    desired_pos = int((target_ratio * n_neg) / (1 - target_ratio))
    if desired_pos <= n_pos:
        return X, y.values

    add_n = desired_pos - n_pos
    sampled_pos_idx = resample(pos_idx, replace=True, n_samples=add_n, random_state=random_state)

    X_extra = X.loc[sampled_pos_idx]
    y_extra = y.loc[sampled_pos_idx]

    X_bal = pd.concat([X, X_extra], axis=0).reset_index(drop=True)
    y_bal = pd.concat([y, y_extra], axis=0).reset_index(drop=True)

    return X_bal, y_bal.values

def best_threshold_by_f1(y_true, probas, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.901, 0.005)

    best_t = 0.50
    best_f1 = -1

    for t in thresholds:
        preds = (probas >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1

def evaluate(y_true, probas, threshold):
    preds = (probas >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

class ThresholdGuessBinarizerSimple(BaseEstimator, TransformerMixin):
    def __init__(self, quantiles=(0.50, 0.70, 0.85, 0.95)):
        self.quantiles = quantiles

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.input_columns_ = X.columns.tolist()
        self.threshold_map_ = {}
        self.output_columns_ = []

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            unique_vals = np.unique(vals)

            if len(unique_vals) <= 1:
                thresholds = []
            elif len(unique_vals) == 2:
                thresholds = [float(unique_vals.max())]
            else:
                qs = np.quantile(vals, self.quantiles)
                thresholds = sorted(set(float(v) for v in qs))

            self.threshold_map_[col] = thresholds

            for t in thresholds:
                self.output_columns_.append(f"{col}__ge__{round(t, 4)}")

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = pd.DataFrame(index=X.index)

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            for t in self.threshold_map_[col]:
                out[f"{col}__ge__{round(t, 4)}"] = (vals >= t).astype(int)

        if out.shape[1] == 0:
            out["dummy_rule"] = 0

        return out

# =========================================================
# 8) MODEL CANDIDATES
# =========================================================
y_train_full = train_df["hot_week"].astype(int).values
pos = int((y_train_full == 1).sum())
neg = int((y_train_full == 0).sum())
scale_pos_weight = neg / max(pos, 1)

model_grid = {
    "logistic_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.2, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.5, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.7, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=1.5, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
    ],

    "xgboost_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=100, max_depth=2, learning_rate=0.05,
                subsample=0.90, colsample_bytree=0.90, min_child_weight=3,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=150, max_depth=2, learning_rate=0.05,
                subsample=0.90, colsample_bytree=0.90, min_child_weight=5,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=200, max_depth=3, learning_rate=0.03,
                subsample=0.90, colsample_bytree=0.90, min_child_weight=5,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=250, max_depth=3, learning_rate=0.05,
                subsample=0.85, colsample_bytree=0.85, min_child_weight=7,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
    ],

    "decision_tree_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=3, min_samples_leaf=30,
                class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=4, min_samples_leaf=40,
                class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=4, min_samples_leaf=50,
                class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=5, min_samples_leaf=60,
                class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
    ],

    "threshold_guess_interpretable": [
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=0.30,
                class_weight="balanced", max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=0.50,
                class_weight="balanced", max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=1.00,
                class_weight="balanced", max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
    ],

    "svm_rbf_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=0.5, gamma="scale",
                class_weight="balanced", probability=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=1.0, gamma="scale",
                class_weight="balanced", probability=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=1.5, gamma="scale",
                class_weight="balanced", probability=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=2.0, gamma="scale",
                class_weight="balanced", probability=True, random_state=RANDOM_STATE
            ))
        ]),
    ],

    "neural_network_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(16,), alpha=0.001, max_iter=800,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(32,), alpha=0.001, max_iter=900,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(32, 16), alpha=0.001, max_iter=1000,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
    ],
}

# =========================================================
# 9) SEARCH LOOP
# =========================================================
all_results = []
best_overall = None

for fs_name, feature_cols in feature_sets.items():
    print(f"\n{'='*70}")
    print("FEATURE SET:", fs_name)
    print("N FEATURES:", len(feature_cols))
    print(feature_cols)

    X_train = train_df[feature_cols].copy()
    y_train = train_df["hot_week"].astype(int).values
    X_val = val_df[feature_cols].copy()
    y_val = val_df["hot_week"].astype(int).values
    X_test = test_df[feature_cols].copy()
    y_test = test_df["hot_week"].astype(int).values

    fs_results = []
    fs_best_models = {}

    for model_name, candidates in model_grid.items():
        best_val_f1 = -1
        best_threshold = 0.50
        best_pipe = None
        best_candidate_num = None
        best_extra = None

        if model_name == "neural_network_black_box":
            oversample_options = [0.25, 0.30, 0.40]
        else:
            oversample_options = [None]

        for i, base_pipe in enumerate(candidates, start=1):
            for osr in oversample_options:
                pipe = clone(base_pipe)

                if model_name == "neural_network_black_box":
                    X_fit, y_fit = oversample_minority(X_train, y_train, target_ratio=osr, random_state=RANDOM_STATE)
                    pipe.fit(X_fit, y_fit)
                else:
                    pipe.fit(X_train, y_train)

                val_proba = pipe.predict_proba(X_val)[:, 1]
                threshold, val_f1 = best_threshold_by_f1(y_val, val_proba)

                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_threshold = threshold
                    best_pipe = pipe
                    best_candidate_num = i
                    best_extra = osr

        X_trval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
        y_trval = np.concatenate([y_train, y_val])

        # refit winning model
        best_pipe = clone(candidates[best_candidate_num - 1])
        if model_name == "neural_network_black_box":
            X_fit, y_fit = oversample_minority(X_trval, y_trval, target_ratio=best_extra, random_state=RANDOM_STATE)
            best_pipe.fit(X_fit, y_fit)
        else:
            best_pipe.fit(X_trval, y_trval)

        test_proba = best_pipe.predict_proba(X_test)[:, 1]
        test_metrics = evaluate(y_test, test_proba, best_threshold)

        row = {
            "feature_set": fs_name,
            "n_features": len(feature_cols),
            "model": model_name,
            "chosen_candidate": best_candidate_num,
            "mlp_oversample_ratio": best_extra,
            "val_best_f1": round(best_val_f1, 4),
            "test_accuracy": round(test_metrics["accuracy"], 4),
            "test_precision": round(test_metrics["precision"], 4),
            "test_recall": round(test_metrics["recall"], 4),
            "test_f1": round(test_metrics["f1"], 4),
            "chosen_threshold": round(best_threshold, 3),
            "tn": test_metrics["tn"],
            "fp": test_metrics["fp"],
            "fn": test_metrics["fn"],
            "tp": test_metrics["tp"],
        }

        fs_results.append(row)
        all_results.append(row)
        fs_best_models[model_name] = (best_pipe, feature_cols)

    fs_results_df = pd.DataFrame(fs_results).sort_values(
        by=["test_f1", "test_recall", "test_precision", "test_accuracy"],
        ascending=False
    ).reset_index(drop=True)

    print("\n=== RESULTS FOR FEATURE SET:", fs_name, "===")
    print(fs_results_df)

# =========================================================
# 10) OVERALL BEST RESULTS
# =========================================================
all_results_df = pd.DataFrame(all_results).sort_values(
    by=["test_f1", "test_recall", "test_precision", "test_accuracy"],
    ascending=False
).reset_index(drop=True)

print(f"\n{'='*70}")
print("=== TOP 15 OVERALL COMBINATIONS ===")
print(all_results_df.head(15))

best_row = all_results_df.iloc[0]
print(f"\n{'='*70}")
print("=== BEST OVERALL RUN ===")
print(best_row)

# =========================================================
# 11) FIT BEST OVERALL AGAIN FOR INTERPRETABILITY PRINTS
# =========================================================
best_feature_set_name = best_row["feature_set"]
best_model_name = best_row["model"]
best_candidate_num = int(best_row["chosen_candidate"])
best_threshold = float(best_row["chosen_threshold"])
best_osr = best_row["mlp_oversample_ratio"]
best_feature_cols = feature_sets[best_feature_set_name]

X_train = train_df[best_feature_cols].copy()
y_train = train_df["hot_week"].astype(int).values
X_val = val_df[best_feature_cols].copy()
y_val = val_df["hot_week"].astype(int).values
X_test = test_df[best_feature_cols].copy()
y_test = test_df["hot_week"].astype(int).values

X_trval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_trval = np.concatenate([y_train, y_val])

best_pipe = clone(model_grid[best_model_name][best_candidate_num - 1])

if best_model_name == "neural_network_black_box":
    X_fit, y_fit = oversample_minority(X_trval, y_trval, target_ratio=best_osr, random_state=RANDOM_STATE)
    best_pipe.fit(X_fit, y_fit)
else:
    best_pipe.fit(X_trval, y_trval)

print(f"\n{'='*70}")
print("=== BEST FEATURE SET USED ===")
print(best_feature_cols)

if best_model_name == "logistic_semi_interpretable":
    log_clf = best_pipe.named_steps["clf"]
    logistic_importance = pd.DataFrame({
        "feature": best_feature_cols,
        "coef": log_clf.coef_[0]
    }).sort_values("coef", key=np.abs, ascending=False)
    print("\n=== LOGISTIC COEFFICIENTS ===")
    print(logistic_importance)

if best_model_name == "decision_tree_interpretable":
    tree_clf = best_pipe.named_steps["clf"]
    tree_importance = pd.DataFrame({
        "feature": best_feature_cols,
        "importance": tree_clf.feature_importances_
    }).sort_values("importance", ascending=False)
    print("\n=== DECISION TREE IMPORTANCE ===")
    print(tree_importance)

if best_model_name == "xgboost_semi_interpretable":
    xgb_clf = best_pipe.named_steps["clf"]
    xgb_importance = pd.DataFrame({
        "feature": best_feature_cols,
        "importance": xgb_clf.feature_importances_
    }).sort_values("importance", ascending=False)
    print("\n=== XGBOOST IMPORTANCE ===")
    print(xgb_importance)

if best_model_name == "threshold_guess_interpretable":
    tg_bin = best_pipe.named_steps["binarizer"]
    tg_clf = best_pipe.named_steps["clf"]
    tg_rules = pd.DataFrame({
        "rule": tg_bin.output_columns_,
        "coef": tg_clf.coef_[0]
    })
    tg_rules = tg_rules[tg_rules["coef"] != 0].sort_values("coef", key=np.abs, ascending=False)
    print("\n=== THRESHOLD RULES ===")
    print(tg_rules.head(30))

Train weeks: 110 Val weeks: 24 Test weeks: 24
Train hot rate: 0.0971
Val hot rate: 0.0861
Test hot rate: 0.0779

FEATURE SET: core_8
N FEATURES: 8
['shootings_count_roll8_mean', 'shootings_count_roll8_sum', 'shootings_count_roll4_sum', 'consecutive_cold_weeks', 'weeks_since_last_hot', 'hot_week_roll8_sum', 'hot_week_lag2', 'any_fatal_week_roll4_sum']

=== RESULTS FOR FEATURE SET: core_8 ===
  feature_set  n_features                          model  chosen_candidate  \
0      core_8           8    logistic_semi_interpretable                 1   
1      core_8           8     xgboost_semi_interpretable                 3   
2      core_8           8    decision_tree_interpretable                 1   
3      core_8           8       neural_network_black_box                 1   
4      core_8           8  threshold_guess_interpretable                 3   
5      core_8           8              svm_rbf_black_box                 1   

   mlp_oversample_ratio  val_best_f1  test_accuracy  test_p

In [32]:
# =========================================================
# NYC PRECINCT-WEEK SHOOTING HOTSPOT PROJECT
# IMPROVED F1 SEARCH + RUNTIMES
# - all 6 models
# - runtime tracking per candidate and per model
# - feature set comparison
# - no file saving
# =========================================================

# If needed in Colab:
# !pip -q install xgboost

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils import resample

from xgboost import XGBClassifier

RANDOM_STATE = 42

# =========================================================
# 1) LOAD DATA
# =========================================================
file_path = "/content/ShootingVictim 2022-2024.csv"
df = pd.read_csv(file_path)

df = df[df["PRECINCT"].notna()].copy()
df["PRECINCT"] = pd.to_numeric(df["PRECINCT"], errors="coerce")
df = df.dropna(subset=["PRECINCT"]).copy()
df["PRECINCT"] = df["PRECINCT"].astype(int)

df["OCCUR_DATE"] = pd.to_datetime(df["OCCUR_DATE"], errors="coerce")
df = df.dropna(subset=["OCCUR_DATE"]).copy()

df["week_start"] = df["OCCUR_DATE"] - pd.to_timedelta(df["OCCUR_DATE"].dt.weekday, unit="D")
df["week_start"] = pd.to_datetime(df["week_start"]).dt.normalize()

numeric_cols = ["victims_in_incident", "incident_murder_flag", "multi_victim_flag"]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["victims_in_incident"] = df["victims_in_incident"].clip(lower=0)
df["incident_murder_flag"] = df["incident_murder_flag"].clip(lower=0, upper=1)
df["multi_victim_flag"] = np.where(df["victims_in_incident"] >= 2, 1, df["multi_victim_flag"])
df["multi_victim_flag"] = pd.to_numeric(df["multi_victim_flag"], errors="coerce").fillna(0).clip(0, 1)

# =========================================================
# 2) INCIDENT LEVEL
# =========================================================
incident_cols = [
    "INCIDENT_KEY",
    "OCCUR_DATE",
    "week_start",
    "PRECINCT",
    "victims_in_incident",
    "incident_murder_flag",
    "multi_victim_flag"
]

inc = (
    df[incident_cols]
    .sort_values(["INCIDENT_KEY", "OCCUR_DATE"])
    .drop_duplicates(subset=["INCIDENT_KEY"])
    .copy()
)

# =========================================================
# 3) WEEKLY AGGREGATION + HOT LABEL
# =========================================================
weekly = (
    inc.groupby(["PRECINCT", "week_start"], as_index=False)
       .agg(
           shootings_count=("INCIDENT_KEY", "nunique"),
           total_victims_week=("victims_in_incident", "sum"),
           max_victims_in_incident=("victims_in_incident", "max"),
           any_fatal_week=("incident_murder_flag", "max"),
           multi_victim_incidents_week=("multi_victim_flag", "sum"),
       )
)

weekly["hot_week"] = np.where(
    (weekly["shootings_count"] >= 2) |
    (weekly["max_victims_in_incident"] >= 2) |
    (weekly["any_fatal_week"] == 1),
    1, 0
)

all_precincts = sorted(inc["PRECINCT"].dropna().unique())
all_weeks = pd.date_range(weekly["week_start"].min(), weekly["week_start"].max(), freq="W-MON")

panel = pd.MultiIndex.from_product(
    [all_precincts, all_weeks],
    names=["PRECINCT", "week_start"]
).to_frame(index=False)

panel = panel.merge(weekly, on=["PRECINCT", "week_start"], how="left")

fill_zero_cols = [
    "shootings_count",
    "total_victims_week",
    "max_victims_in_incident",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week"
]
panel[fill_zero_cols] = panel[fill_zero_cols].fillna(0)
panel = panel.sort_values(["PRECINCT", "week_start"]).reset_index(drop=True)

# =========================================================
# 4) FEATURE ENGINEERING
# =========================================================
def add_lag_features(data, group_col, col, lags=(1, 2, 3, 4)):
    for lag in lags:
        data[f"{col}_lag{lag}"] = data.groupby(group_col)[col].shift(lag)
    return data

def add_roll_features(data, group_col, col, windows=(2, 4, 8)):
    shifted = data.groupby(group_col)[col].shift(1)
    grouped = shifted.groupby(data[group_col])

    for w in windows:
        data[f"{col}_roll{w}_sum"] = (
            grouped.rolling(w, min_periods=1).sum().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_mean"] = (
            grouped.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_max"] = (
            grouped.rolling(w, min_periods=1).max().reset_index(level=0, drop=True)
        )
    return data

base_cols = [
    "shootings_count",
    "total_victims_week",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week",
]

for col in base_cols:
    panel = add_lag_features(panel, "PRECINCT", col, lags=(1, 2, 3, 4))
    panel = add_roll_features(panel, "PRECINCT", col, windows=(2, 4, 8))

for w in [2, 4, 8]:
    panel[f"victims_per_shooting_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"total_victims_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )
    panel[f"fatal_rate_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"any_fatal_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )
    panel[f"multi_victim_rate_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"multi_victim_incidents_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )

panel["weeks_since_last_hot"] = np.nan
panel["consecutive_cold_weeks"] = np.nan

for precinct, idx in panel.groupby("PRECINCT").groups.items():
    idx = list(idx)
    hot_vals = panel.loc[idx, "hot_week"].values

    ws_vals = []
    cold_vals = []

    seen_hot = False
    since_hot = np.nan
    cold_streak = 0

    for i in range(len(hot_vals)):
        if i == 0:
            ws_vals.append(np.nan)
            cold_vals.append(np.nan)
            continue

        if hot_vals[i - 1] == 1:
            since_hot = 0
            seen_hot = True
        elif seen_hot:
            since_hot += 1
        else:
            since_hot = np.nan
        ws_vals.append(since_hot)

        if hot_vals[i - 1] == 0:
            cold_streak += 1
        else:
            cold_streak = 0
        cold_vals.append(cold_streak)

    panel.loc[idx, "weeks_since_last_hot"] = ws_vals
    panel.loc[idx, "consecutive_cold_weeks"] = cold_vals

panel["shooting_hot_interaction"] = panel["shootings_count_roll4_sum"] * panel["hot_week_roll4_sum"]
panel["victim_fatal_interaction"] = panel["total_victims_week_roll4_sum"] * panel["any_fatal_week_roll4_sum"]
panel["shootings_trend_4_vs_8"] = panel["shootings_count_roll4_mean"] - panel["shootings_count_roll8_mean"]
panel["victims_trend_4_vs_8"] = panel["total_victims_week_roll4_mean"] - panel["total_victims_week_roll8_mean"]
panel["hot_trend_4_vs_8"] = panel["hot_week_roll4_mean"] - panel["hot_week_roll8_mean"]

# =========================================================
# 5) FEATURE SETS TO TEST
# =========================================================
feature_sets = {
    "balanced_12": [
        "shootings_count_roll8_mean",
        "shootings_count_roll8_sum",
        "shootings_count_roll4_sum",
        "shootings_count_lag2",
        "consecutive_cold_weeks",
        "weeks_since_last_hot",
        "hot_week_roll8_sum",
        "hot_week_lag1",
        "hot_week_lag2",
        "any_fatal_week_roll4_sum",
        "total_victims_week_roll4_sum",
        "victims_per_shooting_roll4",
    ],

    "strong_16": [
        "shootings_count_roll8_mean",
        "shootings_count_roll8_sum",
        "shootings_count_roll4_sum",
        "shootings_count_lag2",
        "shootings_count_roll4_max",
        "consecutive_cold_weeks",
        "weeks_since_last_hot",
        "hot_week_roll8_sum",
        "hot_week_roll4_sum",
        "hot_week_lag1",
        "hot_week_lag2",
        "any_fatal_week_roll4_sum",
        "total_victims_week_roll4_sum",
        "multi_victim_incidents_week_roll4_sum",
        "victims_per_shooting_roll4",
        "shooting_hot_interaction",
    ],

    "strong_20": [
        "shootings_count_roll8_mean",
        "shootings_count_roll8_sum",
        "shootings_count_roll4_sum",
        "shootings_count_lag2",
        "shootings_count_roll4_max",
        "consecutive_cold_weeks",
        "weeks_since_last_hot",
        "hot_week_roll8_sum",
        "hot_week_roll4_sum",
        "hot_week_lag1",
        "hot_week_lag2",
        "any_fatal_week_roll4_sum",
        "any_fatal_week_roll4_max",
        "total_victims_week_roll4_sum",
        "total_victims_week_roll4_max",
        "multi_victim_incidents_week_lag1",
        "multi_victim_incidents_week_roll4_sum",
        "victims_per_shooting_roll4",
        "fatal_rate_roll4",
        "shooting_hot_interaction",
    ],

    "focused_18": [
        "shootings_count_roll8_mean",
        "shootings_count_roll8_sum",
        "shootings_count_roll4_sum",
        "shootings_count_lag2",
        "shootings_count_roll4_mean",
        "consecutive_cold_weeks",
        "weeks_since_last_hot",
        "hot_week_roll8_sum",
        "hot_week_roll4_sum",
        "hot_week_lag1",
        "hot_week_lag2",
        "any_fatal_week_roll4_sum",
        "total_victims_week_roll4_sum",
        "multi_victim_incidents_week_roll4_sum",
        "victims_per_shooting_roll4",
        "fatal_rate_roll4",
        "multi_victim_rate_roll4",
        "shootings_trend_4_vs_8",
    ],
}

all_needed_features = sorted(set(sum(feature_sets.values(), [])))
model_df = panel[["PRECINCT", "week_start", "hot_week"] + all_needed_features].copy()
model_df[all_needed_features] = model_df[all_needed_features].replace([np.inf, -np.inf], np.nan).fillna(0)

for c in all_needed_features:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0).astype(float)

# =========================================================
# 6) TIME SPLIT
# =========================================================
unique_weeks = sorted(model_df["week_start"].unique())
n_weeks = len(unique_weeks)

train_end = int(n_weeks * 0.70)
val_end = int(n_weeks * 0.85)

train_weeks = unique_weeks[:train_end]
val_weeks = unique_weeks[train_end:val_end]
test_weeks = unique_weeks[val_end:]

train_df = model_df[model_df["week_start"].isin(train_weeks)].copy()
val_df = model_df[model_df["week_start"].isin(val_weeks)].copy()
test_df = model_df[model_df["week_start"].isin(test_weeks)].copy()

print("Train weeks:", len(train_weeks), "Val weeks:", len(val_weeks), "Test weeks:", len(test_weeks))
print("Train hot rate:", round(train_df["hot_week"].mean(), 4))
print("Val hot rate:", round(val_df["hot_week"].mean(), 4))
print("Test hot rate:", round(test_df["hot_week"].mean(), 4))

# =========================================================
# 7) HELPERS
# =========================================================
def oversample_minority(X, y, target_ratio=0.30, random_state=42):
    X = X.reset_index(drop=True).copy()
    y = pd.Series(y).reset_index(drop=True)

    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index

    n_pos = len(pos_idx)
    n_neg = len(neg_idx)

    if n_pos == 0 or n_neg == 0:
        return X, y.values

    desired_pos = int((target_ratio * n_neg) / (1 - target_ratio))
    if desired_pos <= n_pos:
        return X, y.values

    add_n = desired_pos - n_pos
    sampled_pos_idx = resample(pos_idx, replace=True, n_samples=add_n, random_state=random_state)

    X_extra = X.loc[sampled_pos_idx]
    y_extra = y.loc[sampled_pos_idx]

    X_bal = pd.concat([X, X_extra], axis=0).reset_index(drop=True)
    y_bal = pd.concat([y, y_extra], axis=0).reset_index(drop=True)

    return X_bal, y_bal.values

def best_threshold_by_f1(y_true, probas, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.901, 0.005)

    best_t = 0.50
    best_f1 = -1

    for t in thresholds:
        preds = (probas >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1

def evaluate(y_true, probas, threshold):
    preds = (probas >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "threshold": threshold,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

class ThresholdGuessBinarizerSimple(BaseEstimator, TransformerMixin):
    def __init__(self, quantiles=(0.50, 0.70, 0.85, 0.95)):
        self.quantiles = quantiles

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.input_columns_ = X.columns.tolist()
        self.threshold_map_ = {}
        self.output_columns_ = []

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            unique_vals = np.unique(vals)

            if len(unique_vals) <= 1:
                thresholds = []
            elif len(unique_vals) == 2:
                thresholds = [float(unique_vals.max())]
            else:
                qs = np.quantile(vals, self.quantiles)
                thresholds = sorted(set(float(v) for v in qs))

            self.threshold_map_[col] = thresholds
            for t in thresholds:
                self.output_columns_.append(f"{col}__ge__{round(t, 4)}")

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = pd.DataFrame(index=X.index)

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            for t in self.threshold_map_[col]:
                out[f"{col}__ge__{round(t, 4)}"] = (vals >= t).astype(int)

        if out.shape[1] == 0:
            out["dummy_rule"] = 0

        return out

# =========================================================
# 8) MODEL CANDIDATES
# =========================================================
y_train_full = train_df["hot_week"].astype(int).values
pos = int((y_train_full == 1).sum())
neg = int((y_train_full == 0).sum())
scale_pos_weight = neg / max(pos, 1)

model_grid = {
    "logistic_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.15, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.2, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.35, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.5, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
    ],

    "xgboost_semi_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=150, max_depth=2, learning_rate=0.05,
                subsample=0.90, colsample_bytree=0.90, min_child_weight=3,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=200, max_depth=2, learning_rate=0.04,
                subsample=0.90, colsample_bytree=0.90, min_child_weight=5,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=200, max_depth=3, learning_rate=0.03,
                subsample=0.90, colsample_bytree=0.90, min_child_weight=5,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=250, max_depth=3, learning_rate=0.05,
                subsample=0.85, colsample_bytree=0.85, min_child_weight=7,
                reg_lambda=1.5, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
    ],

    "decision_tree_interpretable": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=3, min_samples_leaf=30, class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=4, min_samples_leaf=40, class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=4, min_samples_leaf=50, class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=5, min_samples_leaf=60, class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
    ],

    "threshold_guess_interpretable": [
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=0.30, class_weight="balanced",
                max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=0.50, class_weight="balanced",
                max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=1.00, class_weight="balanced",
                max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
    ],

    "svm_rbf_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=0.5, gamma="scale", class_weight="balanced",
                probability=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=1.0, gamma="scale", class_weight="balanced",
                probability=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=1.5, gamma="scale", class_weight="balanced",
                probability=True, random_state=RANDOM_STATE
            ))
        ]),
    ],

    "neural_network_black_box": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(16,), alpha=0.001, max_iter=800,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(32,), alpha=0.001, max_iter=900,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(32, 16), alpha=0.001, max_iter=1000,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
    ],
}

# =========================================================
# 9) SEARCH LOOP WITH RUNTIMES
# =========================================================
all_results = []
candidate_timing_rows = []

for fs_name, feature_cols in feature_sets.items():
    fs_start = time.perf_counter()

    print(f"\n{'='*78}")
    print("FEATURE SET:", fs_name)
    print("N FEATURES:", len(feature_cols))
    print(feature_cols)

    X_train = train_df[feature_cols].copy()
    y_train = train_df["hot_week"].astype(int).values
    X_val = val_df[feature_cols].copy()
    y_val = val_df["hot_week"].astype(int).values
    X_test = test_df[feature_cols].copy()
    y_test = test_df["hot_week"].astype(int).values

    fs_results = []

    for model_name, candidates in model_grid.items():
        model_start = time.perf_counter()

        best_val_f1 = -1
        best_threshold = 0.50
        best_pipe = None
        best_candidate_num = None
        best_extra = None
        best_candidate_runtime = None

        oversample_options = [None]
        if model_name == "neural_network_black_box":
            oversample_options = [0.25, 0.30, 0.40]

        for i, base_pipe in enumerate(candidates, start=1):
            for osr in oversample_options:
                pipe = clone(base_pipe)

                cand_start = time.perf_counter()

                if model_name == "neural_network_black_box":
                    X_fit, y_fit = oversample_minority(X_train, y_train, target_ratio=osr, random_state=RANDOM_STATE)
                    pipe.fit(X_fit, y_fit)
                else:
                    pipe.fit(X_train, y_train)

                val_proba = pipe.predict_proba(X_val)[:, 1]
                threshold, val_f1 = best_threshold_by_f1(y_val, val_proba)

                cand_runtime = time.perf_counter() - cand_start

                candidate_timing_rows.append({
                    "feature_set": fs_name,
                    "model": model_name,
                    "candidate": i,
                    "mlp_oversample_ratio": osr,
                    "val_best_f1": round(val_f1, 4),
                    "threshold": round(threshold, 3),
                    "runtime_seconds": round(cand_runtime, 3),
                })

                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_threshold = threshold
                    best_pipe = pipe
                    best_candidate_num = i
                    best_extra = osr
                    best_candidate_runtime = cand_runtime

        # refit winner on train+val
        X_trval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
        y_trval = np.concatenate([y_train, y_val])

        refit_start = time.perf_counter()
        best_pipe = clone(candidates[best_candidate_num - 1])

        if model_name == "neural_network_black_box":
            X_fit, y_fit = oversample_minority(X_trval, y_trval, target_ratio=best_extra, random_state=RANDOM_STATE)
            best_pipe.fit(X_fit, y_fit)
        else:
            best_pipe.fit(X_trval, y_trval)

        test_proba = best_pipe.predict_proba(X_test)[:, 1]
        test_metrics = evaluate(y_test, test_proba, best_threshold)
        refit_runtime = time.perf_counter() - refit_start
        total_model_runtime = time.perf_counter() - model_start

        row = {
            "feature_set": fs_name,
            "n_features": len(feature_cols),
            "model": model_name,
            "chosen_candidate": best_candidate_num,
            "mlp_oversample_ratio": best_extra,
            "val_best_f1": round(best_val_f1, 4),
            "test_accuracy": round(test_metrics["accuracy"], 4),
            "test_precision": round(test_metrics["precision"], 4),
            "test_recall": round(test_metrics["recall"], 4),
            "test_f1": round(test_metrics["f1"], 4),
            "chosen_threshold": round(best_threshold, 3),
            "best_candidate_runtime_sec": round(best_candidate_runtime, 3) if best_candidate_runtime is not None else None,
            "refit_plus_test_runtime_sec": round(refit_runtime, 3),
            "total_model_runtime_sec": round(total_model_runtime, 3),
            "tn": test_metrics["tn"],
            "fp": test_metrics["fp"],
            "fn": test_metrics["fn"],
            "tp": test_metrics["tp"],
        }

        fs_results.append(row)
        all_results.append(row)

    fs_runtime = time.perf_counter() - fs_start

    fs_results_df = pd.DataFrame(fs_results).sort_values(
        by=["test_f1", "test_recall", "test_precision", "test_accuracy"],
        ascending=False
    ).reset_index(drop=True)

    print("\n=== RESULTS FOR FEATURE SET:", fs_name, "===")
    print(fs_results_df)

    print("\n=== RUNTIME SUMMARY FOR FEATURE SET:", fs_name, "===")
    print("Total feature-set runtime (seconds):", round(fs_runtime, 3))
    print(fs_results_df[[
        "model",
        "chosen_candidate",
        "mlp_oversample_ratio",
        "best_candidate_runtime_sec",
        "refit_plus_test_runtime_sec",
        "total_model_runtime_sec",
        "val_best_f1",
        "test_f1"
    ]].sort_values("total_model_runtime_sec"))

# =========================================================
# 10) OVERALL BEST RESULTS
# =========================================================
all_results_df = pd.DataFrame(all_results).sort_values(
    by=["test_f1", "test_recall", "test_precision", "test_accuracy"],
    ascending=False
).reset_index(drop=True)

timing_df = pd.DataFrame(candidate_timing_rows).sort_values(
    by=["runtime_seconds"],
    ascending=False
).reset_index(drop=True)

print(f"\n{'='*78}")
print("=== TOP 20 OVERALL COMBINATIONS ===")
print(all_results_df.head(20))

print(f"\n{'='*78}")
print("=== FASTEST / SLOWEST CANDIDATES ===")
print("Top 20 slowest candidate runs:")
print(timing_df.head(20))

print(f"\n{'='*78}")
print("=== BEST OVERALL RUN ===")
print(all_results_df.iloc[0])

# =========================================================
# 11) OPTIONAL: FIT BEST OVERALL AGAIN FOR INTERPRETABILITY
# =========================================================
best_row = all_results_df.iloc[0]
best_feature_set_name = best_row["feature_set"]
best_model_name = best_row["model"]
best_candidate_num = int(best_row["chosen_candidate"])
best_osr = best_row["mlp_oversample_ratio"]
best_feature_cols = feature_sets[best_feature_set_name]

X_train = train_df[best_feature_cols].copy()
y_train = train_df["hot_week"].astype(int).values
X_val = val_df[best_feature_cols].copy()
y_val = val_df["hot_week"].astype(int).values
X_trval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_trval = np.concatenate([y_train, y_val])

best_pipe = clone(model_grid[best_model_name][best_candidate_num - 1])

if best_model_name == "neural_network_black_box":
    X_fit, y_fit = oversample_minority(X_trval, y_trval, target_ratio=best_osr, random_state=RANDOM_STATE)
    best_pipe.fit(X_fit, y_fit)
else:
    best_pipe.fit(X_trval, y_trval)

print(f"\n{'='*78}")
print("=== BEST FEATURE SET USED ===")
print(best_feature_cols)

if best_model_name == "logistic_semi_interpretable":
    log_clf = best_pipe.named_steps["clf"]
    logistic_importance = pd.DataFrame({
        "feature": best_feature_cols,
        "coef": log_clf.coef_[0]
    }).sort_values("coef", key=np.abs, ascending=False)
    print("\n=== LOGISTIC COEFFICIENTS ===")
    print(logistic_importance)

if best_model_name == "decision_tree_interpretable":
    tree_clf = best_pipe.named_steps["clf"]
    tree_importance = pd.DataFrame({
        "feature": best_feature_cols,
        "importance": tree_clf.feature_importances_
    }).sort_values("importance", ascending=False)
    print("\n=== DECISION TREE IMPORTANCE ===")
    print(tree_importance)

if best_model_name == "xgboost_semi_interpretable":
    xgb_clf = best_pipe.named_steps["clf"]
    xgb_importance = pd.DataFrame({
        "feature": best_feature_cols,
        "importance": xgb_clf.feature_importances_
    }).sort_values("importance", ascending=False)
    print("\n=== XGBOOST IMPORTANCE ===")
    print(xgb_importance)

if best_model_name == "threshold_guess_interpretable":
    tg_bin = best_pipe.named_steps["binarizer"]
    tg_clf = best_pipe.named_steps["clf"]
    tg_rules = pd.DataFrame({
        "rule": tg_bin.output_columns_,
        "coef": tg_clf.coef_[0]
    })
    tg_rules = tg_rules[tg_rules["coef"] != 0].sort_values("coef", key=np.abs, ascending=False)
    print("\n=== THRESHOLD RULES ===")
    print(tg_rules.head(30))

Train weeks: 110 Val weeks: 24 Test weeks: 24
Train hot rate: 0.0971
Val hot rate: 0.0861
Test hot rate: 0.0779

FEATURE SET: balanced_12
N FEATURES: 12
['shootings_count_roll8_mean', 'shootings_count_roll8_sum', 'shootings_count_roll4_sum', 'shootings_count_lag2', 'consecutive_cold_weeks', 'weeks_since_last_hot', 'hot_week_roll8_sum', 'hot_week_lag1', 'hot_week_lag2', 'any_fatal_week_roll4_sum', 'total_victims_week_roll4_sum', 'victims_per_shooting_roll4']

=== RESULTS FOR FEATURE SET: balanced_12 ===
   feature_set  n_features                          model  chosen_candidate  \
0  balanced_12          12    logistic_semi_interpretable                 1   
1  balanced_12          12     xgboost_semi_interpretable                 3   
2  balanced_12          12    decision_tree_interpretable                 1   
3  balanced_12          12       neural_network_black_box                 2   
4  balanced_12          12  threshold_guess_interpretable                 3   
5  balanced_12    

In [33]:
# =========================================================
# NYC PRECINCT-WEEK SHOOTING HOTSPOT PROJECT
# CLEAN F1 TUNING VERSION
# - all 6 models
# - cleaner runtime output
# - prints Accuracy / Precision / Recall / F1 / Run Time
# - prints feature importance for interpretable/semi-interpretable models
# - no file saving
# =========================================================

# If needed in Colab:
# !pip -q install xgboost

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils import resample

from xgboost import XGBClassifier

RANDOM_STATE = 42

# =========================================================
# 1) LOAD DATA
# =========================================================
file_path = "/content/ShootingVictim 2022-2024.csv"
df = pd.read_csv(file_path)

df = df[df["PRECINCT"].notna()].copy()
df["PRECINCT"] = pd.to_numeric(df["PRECINCT"], errors="coerce")
df = df.dropna(subset=["PRECINCT"]).copy()
df["PRECINCT"] = df["PRECINCT"].astype(int)

df["OCCUR_DATE"] = pd.to_datetime(df["OCCUR_DATE"], errors="coerce")
df = df.dropna(subset=["OCCUR_DATE"]).copy()

df["week_start"] = df["OCCUR_DATE"] - pd.to_timedelta(df["OCCUR_DATE"].dt.weekday, unit="D")
df["week_start"] = pd.to_datetime(df["week_start"]).dt.normalize()

numeric_cols = ["victims_in_incident", "incident_murder_flag", "multi_victim_flag"]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["victims_in_incident"] = df["victims_in_incident"].clip(lower=0)
df["incident_murder_flag"] = df["incident_murder_flag"].clip(lower=0, upper=1)
df["multi_victim_flag"] = np.where(df["victims_in_incident"] >= 2, 1, df["multi_victim_flag"])
df["multi_victim_flag"] = pd.to_numeric(df["multi_victim_flag"], errors="coerce").fillna(0).clip(0, 1)

# =========================================================
# 2) INCIDENT LEVEL
# =========================================================
incident_cols = [
    "INCIDENT_KEY",
    "OCCUR_DATE",
    "week_start",
    "PRECINCT",
    "victims_in_incident",
    "incident_murder_flag",
    "multi_victim_flag"
]

inc = (
    df[incident_cols]
    .sort_values(["INCIDENT_KEY", "OCCUR_DATE"])
    .drop_duplicates(subset=["INCIDENT_KEY"])
    .copy()
)

# =========================================================
# 3) WEEKLY AGGREGATION + HOT LABEL
# =========================================================
weekly = (
    inc.groupby(["PRECINCT", "week_start"], as_index=False)
       .agg(
           shootings_count=("INCIDENT_KEY", "nunique"),
           total_victims_week=("victims_in_incident", "sum"),
           max_victims_in_incident=("victims_in_incident", "max"),
           any_fatal_week=("incident_murder_flag", "max"),
           multi_victim_incidents_week=("multi_victim_flag", "sum"),
       )
)

weekly["hot_week"] = np.where(
    (weekly["shootings_count"] >= 2) |
    (weekly["max_victims_in_incident"] >= 2) |
    (weekly["any_fatal_week"] == 1),
    1, 0
)

all_precincts = sorted(inc["PRECINCT"].dropna().unique())
all_weeks = pd.date_range(weekly["week_start"].min(), weekly["week_start"].max(), freq="W-MON")

panel = pd.MultiIndex.from_product(
    [all_precincts, all_weeks],
    names=["PRECINCT", "week_start"]
).to_frame(index=False)

panel = panel.merge(weekly, on=["PRECINCT", "week_start"], how="left")

fill_zero_cols = [
    "shootings_count",
    "total_victims_week",
    "max_victims_in_incident",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week"
]
panel[fill_zero_cols] = panel[fill_zero_cols].fillna(0)
panel = panel.sort_values(["PRECINCT", "week_start"]).reset_index(drop=True)

# =========================================================
# 4) FEATURE ENGINEERING
# =========================================================
def add_lag_features(data, group_col, col, lags=(1, 2, 3, 4)):
    for lag in lags:
        data[f"{col}_lag{lag}"] = data.groupby(group_col)[col].shift(lag)
    return data

def add_roll_features(data, group_col, col, windows=(2, 4, 8)):
    shifted = data.groupby(group_col)[col].shift(1)
    grouped = shifted.groupby(data[group_col])

    for w in windows:
        data[f"{col}_roll{w}_sum"] = (
            grouped.rolling(w, min_periods=1).sum().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_mean"] = (
            grouped.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
        )
        data[f"{col}_roll{w}_max"] = (
            grouped.rolling(w, min_periods=1).max().reset_index(level=0, drop=True)
        )
    return data

base_cols = [
    "shootings_count",
    "total_victims_week",
    "any_fatal_week",
    "multi_victim_incidents_week",
    "hot_week",
]

for col in base_cols:
    panel = add_lag_features(panel, "PRECINCT", col, lags=(1, 2, 3, 4))
    panel = add_roll_features(panel, "PRECINCT", col, windows=(2, 4, 8))

for w in [2, 4, 8]:
    panel[f"victims_per_shooting_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"total_victims_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )
    panel[f"fatal_rate_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"any_fatal_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )
    panel[f"multi_victim_rate_roll{w}"] = np.where(
        panel[f"shootings_count_roll{w}_sum"] > 0,
        panel[f"multi_victim_incidents_week_roll{w}_sum"] / panel[f"shootings_count_roll{w}_sum"],
        0
    )

panel["weeks_since_last_hot"] = np.nan
panel["consecutive_cold_weeks"] = np.nan

for precinct, idx in panel.groupby("PRECINCT").groups.items():
    idx = list(idx)
    hot_vals = panel.loc[idx, "hot_week"].values

    ws_vals = []
    cold_vals = []

    seen_hot = False
    since_hot = np.nan
    cold_streak = 0

    for i in range(len(hot_vals)):
        if i == 0:
            ws_vals.append(np.nan)
            cold_vals.append(np.nan)
            continue

        if hot_vals[i - 1] == 1:
            since_hot = 0
            seen_hot = True
        elif seen_hot:
            since_hot += 1
        else:
            since_hot = np.nan
        ws_vals.append(since_hot)

        if hot_vals[i - 1] == 0:
            cold_streak += 1
        else:
            cold_streak = 0
        cold_vals.append(cold_streak)

    panel.loc[idx, "weeks_since_last_hot"] = ws_vals
    panel.loc[idx, "consecutive_cold_weeks"] = cold_vals

panel["shooting_hot_interaction"] = panel["shootings_count_roll4_sum"] * panel["hot_week_roll4_sum"]
panel["victim_fatal_interaction"] = panel["total_victims_week_roll4_sum"] * panel["any_fatal_week_roll4_sum"]
panel["shootings_trend_4_vs_8"] = panel["shootings_count_roll4_mean"] - panel["shootings_count_roll8_mean"]
panel["victims_trend_4_vs_8"] = panel["total_victims_week_roll4_mean"] - panel["total_victims_week_roll8_mean"]
panel["hot_trend_4_vs_8"] = panel["hot_week_roll4_mean"] - panel["hot_week_roll8_mean"]

# =========================================================
# 5) FEATURE SETS
# =========================================================
feature_sets = {
    "balanced_12": [
        "shootings_count_roll8_mean",
        "shootings_count_roll8_sum",
        "shootings_count_roll4_sum",
        "shootings_count_lag2",
        "consecutive_cold_weeks",
        "weeks_since_last_hot",
        "hot_week_roll8_sum",
        "hot_week_lag1",
        "hot_week_lag2",
        "any_fatal_week_roll4_sum",
        "total_victims_week_roll4_sum",
        "victims_per_shooting_roll4",
    ],

    "strong_16": [
        "shootings_count_roll8_mean",
        "shootings_count_roll8_sum",
        "shootings_count_roll4_sum",
        "shootings_count_lag2",
        "shootings_count_roll4_max",
        "consecutive_cold_weeks",
        "weeks_since_last_hot",
        "hot_week_roll8_sum",
        "hot_week_roll4_sum",
        "hot_week_lag1",
        "hot_week_lag2",
        "any_fatal_week_roll4_sum",
        "total_victims_week_roll4_sum",
        "multi_victim_incidents_week_roll4_sum",
        "victims_per_shooting_roll4",
        "shooting_hot_interaction",
    ],

    "focused_18": [
        "shootings_count_roll8_mean",
        "shootings_count_roll8_sum",
        "shootings_count_roll4_sum",
        "shootings_count_lag2",
        "shootings_count_roll4_mean",
        "shootings_count_roll4_max",
        "consecutive_cold_weeks",
        "weeks_since_last_hot",
        "hot_week_roll8_sum",
        "hot_week_roll4_sum",
        "hot_week_lag1",
        "hot_week_lag2",
        "any_fatal_week_roll4_sum",
        "total_victims_week_roll4_sum",
        "multi_victim_incidents_week_roll4_sum",
        "victims_per_shooting_roll4",
        "fatal_rate_roll4",
        "shootings_trend_4_vs_8",
    ],
}

all_needed_features = sorted(set(sum(feature_sets.values(), [])))
model_df = panel[["PRECINCT", "week_start", "hot_week"] + all_needed_features].copy()
model_df[all_needed_features] = model_df[all_needed_features].replace([np.inf, -np.inf], np.nan).fillna(0)

for c in all_needed_features:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0).astype(float)

# =========================================================
# 6) TIME SPLIT
# =========================================================
unique_weeks = sorted(model_df["week_start"].unique())
n_weeks = len(unique_weeks)

train_end = int(n_weeks * 0.70)
val_end = int(n_weeks * 0.85)

train_weeks = unique_weeks[:train_end]
val_weeks = unique_weeks[train_end:val_end]
test_weeks = unique_weeks[val_end:]

train_df = model_df[model_df["week_start"].isin(train_weeks)].copy()
val_df = model_df[model_df["week_start"].isin(val_weeks)].copy()
test_df = model_df[model_df["week_start"].isin(test_weeks)].copy()

print("Train weeks:", len(train_weeks), "Val weeks:", len(val_weeks), "Test weeks:", len(test_weeks))
print("Train hot rate:", round(train_df["hot_week"].mean(), 4))
print("Val hot rate:", round(val_df["hot_week"].mean(), 4))
print("Test hot rate:", round(test_df["hot_week"].mean(), 4))

# =========================================================
# 7) HELPERS
# =========================================================
def oversample_minority(X, y, target_ratio=0.30, random_state=42):
    X = X.reset_index(drop=True).copy()
    y = pd.Series(y).reset_index(drop=True)

    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index

    n_pos = len(pos_idx)
    n_neg = len(neg_idx)

    if n_pos == 0 or n_neg == 0:
        return X, y.values

    desired_pos = int((target_ratio * n_neg) / (1 - target_ratio))
    if desired_pos <= n_pos:
        return X, y.values

    add_n = desired_pos - n_pos
    sampled_pos_idx = resample(pos_idx, replace=True, n_samples=add_n, random_state=random_state)

    X_extra = X.loc[sampled_pos_idx]
    y_extra = y.loc[sampled_pos_idx]

    X_bal = pd.concat([X, X_extra], axis=0).reset_index(drop=True)
    y_bal = pd.concat([y, y_extra], axis=0).reset_index(drop=True)

    return X_bal, y_bal.values

def best_threshold_by_f1(y_true, probas, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.901, 0.005)

    best_t = 0.50
    best_f1 = -1

    for t in thresholds:
        preds = (probas >= t).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t, best_f1

def evaluate(y_true, probas, threshold):
    preds = (probas >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()

    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

class ThresholdGuessBinarizerSimple(BaseEstimator, TransformerMixin):
    def __init__(self, quantiles=(0.50, 0.70, 0.85, 0.95)):
        self.quantiles = quantiles

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.input_columns_ = X.columns.tolist()
        self.threshold_map_ = {}
        self.output_columns_ = []

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            unique_vals = np.unique(vals)

            if len(unique_vals) <= 1:
                thresholds = []
            elif len(unique_vals) == 2:
                thresholds = [float(unique_vals.max())]
            else:
                qs = np.quantile(vals, self.quantiles)
                thresholds = sorted(set(float(v) for v in qs))

            self.threshold_map_[col] = thresholds
            for t in thresholds:
                self.output_columns_.append(f"{col}__ge__{round(t, 4)}")

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        out = pd.DataFrame(index=X.index)

        for col in self.input_columns_:
            vals = pd.to_numeric(X[col], errors="coerce").fillna(0).astype(float)
            for t in self.threshold_map_[col]:
                out[f"{col}__ge__{round(t, 4)}"] = (vals >= t).astype(int)

        if out.shape[1] == 0:
            out["dummy_rule"] = 0

        return out

# =========================================================
# 8) MODELS
# =========================================================
y_train_full = train_df["hot_week"].astype(int).values
pos = int((y_train_full == 1).sum())
neg = int((y_train_full == 0).sum())
scale_pos_weight = neg / max(pos, 1)

model_grid = {
    "logistic": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.15, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.20, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.35, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=0.50, class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE))
        ]),
    ],

    "xgboost": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=150, max_depth=2, learning_rate=0.05,
                subsample=0.90, colsample_bytree=0.90, min_child_weight=3,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=200, max_depth=2, learning_rate=0.04,
                subsample=0.90, colsample_bytree=0.90, min_child_weight=5,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", XGBClassifier(
                n_estimators=200, max_depth=3, learning_rate=0.03,
                subsample=0.90, colsample_bytree=0.90, min_child_weight=5,
                reg_lambda=1.0, objective="binary:logistic", eval_metric="logloss",
                random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight
            ))
        ]),
    ],

    "decision_tree": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=3, min_samples_leaf=30, class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=4, min_samples_leaf=40, class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("clf", DecisionTreeClassifier(
                max_depth=5, min_samples_leaf=60, class_weight="balanced", random_state=RANDOM_STATE
            ))
        ]),
    ],

    "threshold_guess": [
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=0.30,
                class_weight="balanced", max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=0.50,
                class_weight="balanced", max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("binarizer", ThresholdGuessBinarizerSimple(quantiles=(0.50, 0.70, 0.85, 0.95))),
            ("clf", LogisticRegression(
                penalty="l1", solver="liblinear", C=1.00,
                class_weight="balanced", max_iter=4000, random_state=RANDOM_STATE
            ))
        ]),
    ],

    "svm_rbf": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=0.5, gamma="scale",
                class_weight="balanced", probability=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=1.0, gamma="scale",
                class_weight="balanced", probability=True, random_state=RANDOM_STATE
            ))
        ]),
    ],

    "mlp": [
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(16,), alpha=0.001, max_iter=800,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(32,), alpha=0.001, max_iter=900,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
        Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", StandardScaler()),
            ("clf", MLPClassifier(
                hidden_layer_sizes=(32, 16), alpha=0.001, max_iter=1000,
                early_stopping=True, random_state=RANDOM_STATE
            ))
        ]),
    ],
}

# =========================================================
# 9) SEARCH + CLEAN RESULTS
# =========================================================
all_results = []
best_model_objects = {}

for fs_name, feature_cols in feature_sets.items():
    print("\n" + "=" * 78)
    print("FEATURE SET:", fs_name)
    print("N FEATURES:", len(feature_cols))
    print(feature_cols)

    X_train = train_df[feature_cols].copy()
    y_train = train_df["hot_week"].astype(int).values
    X_val = val_df[feature_cols].copy()
    y_val = val_df["hot_week"].astype(int).values
    X_test = test_df[feature_cols].copy()
    y_test = test_df["hot_week"].astype(int).values

    fs_rows = []

    for model_name, candidates in model_grid.items():
        model_start = time.perf_counter()

        best_val_f1 = -1
        best_threshold = None
        best_candidate_num = None
        best_pipe = None
        best_osr = None

        oversample_options = [None]
        if model_name == "mlp":
            oversample_options = [0.25, 0.30, 0.40]

        for i, base_pipe in enumerate(candidates, start=1):
            for osr in oversample_options:
                pipe = clone(base_pipe)

                if model_name == "mlp":
                    X_fit, y_fit = oversample_minority(X_train, y_train, target_ratio=osr, random_state=RANDOM_STATE)
                    pipe.fit(X_fit, y_fit)
                else:
                    pipe.fit(X_train, y_train)

                val_proba = pipe.predict_proba(X_val)[:, 1]
                threshold, val_f1 = best_threshold_by_f1(y_val, val_proba)

                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_threshold = threshold
                    best_candidate_num = i
                    best_pipe = pipe
                    best_osr = osr

        # Refit winner on train+val
        X_trval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
        y_trval = np.concatenate([y_train, y_val])

        final_pipe = clone(candidates[best_candidate_num - 1])

        if model_name == "mlp":
            X_fit, y_fit = oversample_minority(X_trval, y_trval, target_ratio=best_osr, random_state=RANDOM_STATE)
            final_pipe.fit(X_fit, y_fit)
        else:
            final_pipe.fit(X_trval, y_trval)

        test_proba = final_pipe.predict_proba(X_test)[:, 1]
        test_metrics = evaluate(y_test, test_proba, best_threshold)

        runtime_sec = time.perf_counter() - model_start

        fs_rows.append({
            "Feature Set": fs_name,
            "Model": model_name,
            "Candidate": best_candidate_num,
            "MLP Oversample": best_osr,
            "Accuracy %": round(test_metrics["accuracy"] * 100, 2),
            "Precision %": round(test_metrics["precision"] * 100, 2),
            "Recall %": round(test_metrics["recall"] * 100, 2),
            "F1 %": round(test_metrics["f1"] * 100, 2),
            "Threshold": round(best_threshold, 3),
            "Run Time (sec)": round(runtime_sec, 3),
            "Val F1 %": round(best_val_f1 * 100, 2),
            "TN": test_metrics["tn"],
            "FP": test_metrics["fp"],
            "FN": test_metrics["fn"],
            "TP": test_metrics["tp"],
        })

        all_results.append(fs_rows[-1])
        best_model_objects[(fs_name, model_name)] = (final_pipe, feature_cols)

    fs_results_df = pd.DataFrame(fs_rows).sort_values(
        by=["F1 %", "Recall %", "Precision %", "Accuracy %"],
        ascending=False
    ).reset_index(drop=True)

    print("\n=== RESULTS ===")
    print(fs_results_df[[
        "Model", "Accuracy %", "Precision %", "Recall %", "F1 %",
        "Run Time (sec)", "Threshold", "Candidate", "MLP Oversample"
    ]])

overall_df = pd.DataFrame(all_results).sort_values(
    by=["F1 %", "Recall %", "Precision %", "Accuracy %"],
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 78)
print("=== TOP 15 OVERALL RUNS ===")
print(overall_df[[
    "Feature Set", "Model", "Accuracy %", "Precision %", "Recall %", "F1 %",
    "Run Time (sec)", "Threshold", "Candidate", "MLP Oversample"
]].head(15))

best_row = overall_df.iloc[0]
print("\n" + "=" * 78)
print("=== BEST OVERALL RUN ===")
print(best_row)

# =========================================================
# 10) FEATURE IMPORTANCE / RULES FOR BEST RUN
# =========================================================
best_fs = best_row["Feature Set"]
best_model = best_row["Model"]
best_pipe, best_features = best_model_objects[(best_fs, best_model)]

print("\n" + "=" * 78)
print("=== BEST FEATURE SET ===")
print(best_features)

if best_model == "logistic":
    clf = best_pipe.named_steps["clf"]
    imp_df = pd.DataFrame({
        "Feature": best_features,
        "Importance": clf.coef_[0]
    }).sort_values("Importance", key=np.abs, ascending=False)

    print("\n=== LOGISTIC FEATURE IMPORTANCE ===")
    print(imp_df)

elif best_model == "decision_tree":
    clf = best_pipe.named_steps["clf"]
    imp_df = pd.DataFrame({
        "Feature": best_features,
        "Importance": clf.feature_importances_
    }).sort_values("Importance", ascending=False)

    print("\n=== DECISION TREE FEATURE IMPORTANCE ===")
    print(imp_df)

elif best_model == "xgboost":
    clf = best_pipe.named_steps["clf"]
    imp_df = pd.DataFrame({
        "Feature": best_features,
        "Importance": clf.feature_importances_
    }).sort_values("Importance", ascending=False)

    print("\n=== XGBOOST FEATURE IMPORTANCE ===")
    print(imp_df)

elif best_model == "threshold_guess":
    tg_bin = best_pipe.named_steps["binarizer"]
    tg_clf = best_pipe.named_steps["clf"]

    rules_df = pd.DataFrame({
        "Rule": tg_bin.output_columns_,
        "Importance": tg_clf.coef_[0]
    })

    rules_df = rules_df[rules_df["Importance"] != 0].sort_values(
        "Importance", key=np.abs, ascending=False
    )

    print("\n=== THRESHOLD GUESS RULE IMPORTANCE ===")
    print(rules_df.head(30))

# =========================================================
# 11) FEATURE IMPORTANCE FOR TOP LOGISTIC + TOP XGBOOST
# =========================================================
top_logistic = overall_df[overall_df["Model"] == "logistic"].head(1)
top_xgb = overall_df[overall_df["Model"] == "xgboost"].head(1)

if len(top_logistic) > 0:
    row = top_logistic.iloc[0]
    pipe, feats = best_model_objects[(row["Feature Set"], "logistic")]
    clf = pipe.named_steps["clf"]

    imp_df = pd.DataFrame({
        "Feature": feats,
        "Importance": clf.coef_[0]
    }).sort_values("Importance", key=np.abs, ascending=False)

    print("\n" + "=" * 78)
    print("=== TOP LOGISTIC RUN FEATURE IMPORTANCE ===")
    print("Feature Set:", row["Feature Set"])
    print(imp_df)

if len(top_xgb) > 0:
    row = top_xgb.iloc[0]
    pipe, feats = best_model_objects[(row["Feature Set"], "xgboost")]
    clf = pipe.named_steps["clf"]

    imp_df = pd.DataFrame({
        "Feature": feats,
        "Importance": clf.feature_importances_
    }).sort_values("Importance", ascending=False)

    print("\n" + "=" * 78)
    print("=== TOP XGBOOST RUN FEATURE IMPORTANCE ===")
    print("Feature Set:", row["Feature Set"])
    print(imp_df)

Train weeks: 110 Val weeks: 24 Test weeks: 24
Train hot rate: 0.0971
Val hot rate: 0.0861
Test hot rate: 0.0779

FEATURE SET: balanced_12
N FEATURES: 12
['shootings_count_roll8_mean', 'shootings_count_roll8_sum', 'shootings_count_roll4_sum', 'shootings_count_lag2', 'consecutive_cold_weeks', 'weeks_since_last_hot', 'hot_week_roll8_sum', 'hot_week_lag1', 'hot_week_lag2', 'any_fatal_week_roll4_sum', 'total_victims_week_roll4_sum', 'victims_per_shooting_roll4']

=== RESULTS ===
             Model  Accuracy %  Precision %  Recall %   F1 %  Run Time (sec)  \
0         logistic       88.16        31.50     44.37  36.84           2.464   
1          xgboost       88.49        31.91     42.25  36.36           1.671   
2    decision_tree       86.24        27.39     46.48  34.46           1.108   
3              mlp       86.18        26.89     45.07  33.68          10.921   
4  threshold_guess       87.28        27.94     40.14  32.95           1.660   
5          svm_rbf       78.51        19.